# Leaderboard Aggregation — All Layers & Forecasting Horizons

This notebook **reads the saved leaderboard CSV files produced by the Phase-05 model notebooks** and builds clean DataFrames for reporting/publication.

Expected stable artifact structure from the Phase-05 notebook:

```text
Artifacts/
└── results/
    └── {SYMBOL}/
        └── {LAYER}/
            └── h{HORIZON}d/
                └── leaderboard.csv
```

The Phase-05 notebook explicitly writes `leaderboard.csv` under `Artifacts/results/{symbol}/{layer}/h{horizon}d/`. This aggregation notebook therefore does **not retrain any model**; it only reads and reshapes the saved results.


## 1. Configuration

Change only `GDRIVE_BASE`, `SYMBOLS`, `HORIZONS`, and (if necessary) `LAYER_CONFIG`.

`LAYER_CONFIG` deliberately supports both the ternary and binary direction outputs present in the Phase-05 notebook, plus the two regression layers used by the overall project.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1a. Publication-quality plot style (new)

Every figure produced below now follows one consistent, paper-ready style, instead of each
plotting cell setting its own ad-hoc look:

- **Colorblind-safe palette** (Okabe–Ito, the standard 8-color palette recommended for
  scientific figures) is used for *every* categorical grouping in the notebook — model
  category, task, and feature domain all draw from the same 8 colors, so a reader can
  learn the color key once and reuse it across every plot.
- **Serif font, bold titles, no top/right spines, light gridlines** — matches the look of
  figures in most journals/conference templates (IEEE, Elsevier, Springer, etc.).
- **Every figure is saved twice**: a 300 DPI PNG (fine for slides, Word, quick viewing) and
  a matching vector **PDF** (best for LaTeX / high-resolution print — it can be rescaled
  without pixelation). A single `save_fig()` helper does this everywhere so filenames and
  folders stay consistent.

Nothing about the *data* or the aggregation logic changes here — this only touches how
plots look and where they're saved.

In [ ]:
# ============================================================
# 1a. Global publication-quality Matplotlib style
# ============================================================
import matplotlib.pyplot as plt
import matplotlib as mpl

# Okabe-Ito palette: colorblind-safe, the standard choice for scientific figures.
OKABE_ITO = [
    "#0072B2",  # blue
    "#E69F00",  # orange
    "#009E73",  # green
    "#D55E00",  # vermillion
    "#CC79A7",  # pink
    "#56B4E9",  # sky blue
    "#F0E442",  # yellow
    "#000000",  # black
]

PLOT_DPI = 300  # Standard minimum resolution required by most journals/conferences.

mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": PLOT_DPI,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "Liberation Serif"],
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.frameon": True,
    "legend.framealpha": 0.85,
    "legend.edgecolor": "none",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.prop_cycle": mpl.cycler(color=OKABE_ITO),
    "savefig.bbox": "tight",
})


def save_fig(fig, root, name, dpi=PLOT_DPI):
    """Save a figure as both PNG (raster, 300 DPI) and PDF (vector) into `root`,
    under the same base filename. Use this instead of a bare `plt.savefig(...)`
    call so every figure in the notebook ends up in a consistent format/location.
    Returns (png_path, pdf_path).
    """
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    png_path = root / f"{name}.png"
    pdf_path = root / f"{name}.pdf"
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"Saved: {png_path.name}  +  {pdf_path.name}")
    return png_path, pdf_path


print("Publication plot style active.")
print("Palette (Okabe-Ito):", OKABE_ITO)


In [ ]:
# ------------------------------------------------------------------
# Project root
# ------------------------------------------------------------------
GDRIVE_BASE = '/content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107'
ARTIFACT_ROOT = Path(GDRIVE_BASE) / 'Artifacts'
RESULTS_ROOT = ARTIFACT_ROOT / 'results'

# Run one or several symbols. Multiple symbols are stacked as an outer row
# level (Symbol -> Model) in every publication table, matching the layout
# used for reporting, e.g. SYMBOLS = ['NIFTY', 'BANKNIFTY', 'NIFTYNEXT50'].
SYMBOLS = ['NIFTY50', 'BANKNIFTY', 'NIFTYNEXT50']

# Horizons used throughout the project.
HORIZONS = [1, 3, 5, 10, 21]

# Layer names written by the Phase-05 notebook / expected by the project.
LAYER_CONFIG = {
    'L1_1_TRI': {
        'label': 'Layer 1.1 — Ternary Regime Detection',
        'task': 'classification',
        'primary_metrics': ['accuracy', 'f1_macro'],
    },
    'L1_1_DIR': {
        'label': 'Layer 1.1 — Binary Direction Detection',
        'task': 'classification',
        'primary_metrics': ['accuracy', 'f1_macro'],
    },
    'L1_2_MAXDRAW': {
        'label': 'Layer 1.2 — Volatility & Risk Forecasting',
        'task': 'regression',
        'primary_metrics': ['rmse', 'directional_acc'],
    },
    'L1_2_MAXRET': {
        'label': 'Layer 1.2 — Volatility & Risk Forecasting',
        'task': 'regression',
        'primary_metrics': ['rmse', 'directional_acc'],
    },
    'L1_2_RVOL': {
        'label': 'Layer 1.2 — Volatility & Risk Forecasting',
        'task': 'regression',
        'primary_metrics': ['rmse', 'directional_acc'],
    },
    'L1_3_PCT': {
        'label': 'Layer 1.3 — Direction & Return Forecasting',
        'task': 'regression',
        'primary_metrics': ['rmse', 'directional_acc'],
    },
    'L1_3_RET': {
        'label': 'Layer 1.3 — Direction & Return Forecasting',
        'task': 'regression',
        'primary_metrics': ['rmse', 'directional_acc'],
    },
}

print('RESULTS_ROOT:', RESULTS_ROOT)
print('Symbols:', SYMBOLS)
print('Horizons:', HORIZONS)

### Model categories & exclusions

Models are grouped into **Machine Learning**, **Deep Learning**, and **Hybrid and Ensemble** models. Every leaderboard/publication table below orders models by this category (matching the reporting layout), and the simple statistical baselines `Naive` and `ARIMA` are excluded from all tables.

In [ ]:
# ------------------------------------------------------------------
# Model categorization & ordering (for reporting)
# ------------------------------------------------------------------
MODEL_CATEGORIES = {
    'Machine Learning Models': [
        'CatBoost_Tuned',
        'Gaussian_NB',
        'Gradient_Boosting',
        'KNN',
        'LightGBM_Tuned',
        'Linear_Logistic',
        'MLP',
        'Random_Forest',
        'SVM',
        'XGBoost_Tuned',
    ],
    'Deep Learning Models': [
        'Attention_LSTM',
        'Bidirectional_LSTM',
        'GRU',
        'LSTM',
    ],
    'Hybrid and Ensemble Models': [
        'AttnLSTM_XGBoost',
        'Hybrid_AttnLSTM_XGBoost',
        'MultiParadigm_MetaStacker',
        'MultiParadigm_Stacker',
        'Stacking_Ensemble',
        'Voting_Ensemble',
    ],
}

# Flattened order used to sort models (by category, then as listed) in every
# leaderboard / publication table produced below.
MODEL_ORDER = [m for group in MODEL_CATEGORIES.values() for m in group]
MODEL_RANK = {m: i for i, m in enumerate(MODEL_ORDER)}

# Reverse lookup: model name -> its category / "Model Type" label, so
# publication tables can show the category headers exactly like the
# reference table (Machine Learning Models / Deep Learning Models /
# Hybrid and Ensemble Models). Models not found default to 'Other'.
MODEL_TO_TYPE = {m: cat for cat, models in MODEL_CATEGORIES.items() for m in models}

# Models to drop from every table (baselines that are no longer part of the
# reported comparison). Matching is case-insensitive and substring-based, so
# naming variants such as 'Naive_Baseline' or 'ARIMA_Model' are also caught.
EXCLUDED_MODELS = {'Naive', 'ARIMA'}

print(f'Model order ({len(MODEL_ORDER)} models):')
for cat, models in MODEL_CATEGORIES.items():
    print(f'  {cat}: {models}')
print('Excluded models (dropped everywhere):', sorted(EXCLUDED_MODELS))


In [ ]:
MODEL_TO_TYPE

### Model-category color coding (new)

Every plot below that shows individual models colors them by **category** — Machine
Learning / Deep Learning / Hybrid & Ensemble — using the first three colors of the
Okabe–Ito palette. This is the *same* three colors in the forest plots, the best-model
heatmap, the selection-frequency chart, and the combined dashboard, so a reader only has
to learn the color key once.

In [ ]:
CATEGORY_COLORS = {
    'Machine Learning Models': OKABE_ITO[0],
    'Deep Learning Models': OKABE_ITO[1],
    'Hybrid and Ensemble Models': OKABE_ITO[2],
    'Other': OKABE_ITO[7],
}

MODEL_COLORS = {m: CATEGORY_COLORS[cat] for cat, models in MODEL_CATEGORIES.items() for m in models}


def color_for_model(model_name):
    """Category color for a model name; falls back to the 'Other' color for
    any model not listed in MODEL_CATEGORIES (e.g. a new model added later)."""
    return MODEL_COLORS.get(model_name, CATEGORY_COLORS['Other'])


def category_legend_handles():
    """Reusable legend handles for the three model categories (used by every
    model-colored plot below, so the legend is identical everywhere)."""
    return [
        plt.Line2D([0], [0], marker='o', linestyle='', color=c, label=cat, markersize=8)
        for cat, c in CATEGORY_COLORS.items() if cat != 'Other'
    ]


print('Category colors:', CATEGORY_COLORS)


## 2. Discover saved leaderboard files

The loader first checks the **stable Artifact tree**, which is the preferred source because it is date-independent. It also optionally searches the timestamped `Models/Unified_*` tree so older runs can still be aggregated.

In [ ]:
def discover_stable_files(symbols=SYMBOLS, horizons=HORIZONS):
    rows = []
    for symbol in symbols:
        for layer in LAYER_CONFIG:
            for h in horizons:
                p = RESULTS_ROOT / symbol / layer / f'h{h}d' / 'leaderboard.csv'
                rows.append({
                    'symbol': symbol,
                    'layer': layer,
                    'horizon': h,
                    'path': str(p),
                    'exists': p.exists(),
                    'source': 'stable_artifact',
                })
    return pd.DataFrame(rows)

file_inventory = discover_stable_files()
display(file_inventory[file_inventory['exists']].copy().head())

print(f'Found {file_inventory["exists"].sum()} stable leaderboard files.')
print(f'Missing {len(file_inventory) - file_inventory["exists"].sum()} expected files.')

## 3. Read and standardize all leaderboard CSVs

Each source CSV normally contains one row per **model × split**, with `train`, `val`, `train_val`, and/or `test` rows. The aggregation layer preserves the raw long-format data and adds standardized metadata.

**Important:** no metric is recomputed here. Values are read exactly as saved by the model notebook.

In [ ]:
def read_leaderboard_file(path, source=None, symbol=None, layer=None, horizon=None):
    df = pd.read_csv(path)

    # Prefer metadata already stored inside the CSV; otherwise use path metadata.
    if 'symbol' not in df.columns or df['symbol'].isna().all():
        df['symbol'] = symbol
    if 'layer' not in df.columns or df['layer'].isna().all():
        df['layer'] = layer
    if 'horizon' not in df.columns or df['horizon'].isna().all():
        df['horizon'] = horizon

    df['source_file'] = str(path)
    df['source_type'] = source
    return df


def flag_degenerate_rows(df):
    """Flag rows with the exact degenerate/overfit signature:

        classification:  accuracy == 1  AND  f1_macro == 0
        regression:       rmse == 0      AND  directional_acc == 1

    Both combinations are mathematically pathological (perfect accuracy with
    zero macro-F1, or zero error with perfect directional accuracy) and are
    a reliable sign of a broken evaluation row (e.g. a class missing from
    the TEST fold, or a constant/degenerate predictor) rather than a
    genuinely good model. These rows are dropped everywhere downstream so
    they can never win a leaderboard, heatmap cell, or forest plot.
    """
    cond_cls = pd.Series(False, index=df.index)
    if 'accuracy' in df.columns and 'f1_macro' in df.columns:
        cond_cls = (df['accuracy'] == 1) & (df['f1_macro'] == 0)

    cond_reg = pd.Series(False, index=df.index)
    if 'rmse' in df.columns and 'directional_acc' in df.columns:
        cond_reg = (df['rmse'] == 0) & (df['directional_acc'] == 1)

    return (cond_cls | cond_reg).fillna(False)


def load_all_leaderboards():
    frames = []

    stable = file_inventory[file_inventory['exists']].copy()
    for _, r in stable.iterrows():
        frames.append(read_leaderboard_file(
            r['path'], r['source'], r['symbol'], r['layer'], int(r['horizon'])
        ))

    if not frames:
        return pd.DataFrame()

    out = pd.concat(frames, ignore_index=True, sort=False)
    out['horizon'] = pd.to_numeric(out['horizon'], errors='coerce').astype('Int64')

    # Drop excluded baseline models (e.g. Naive, ARIMA) from every downstream table.
    if not out.empty and 'model' in out.columns:
        is_excluded = out['model'].astype(str).str.lower().apply(
            lambda m: any(ex.lower() in m for ex in EXCLUDED_MODELS)
        )
        n_dropped = int(is_excluded.sum())
        if n_dropped:
            print(f'Dropping {n_dropped} rows for excluded models: {sorted(out.loc[is_excluded, "model"].unique())}')
        out = out[~is_excluded].reset_index(drop=True)

    # Drop degenerate / overfit rows (accuracy=1 & f1_macro=0, or rmse=0 & DA=1).
    if not out.empty:
        is_degenerate = flag_degenerate_rows(out)
        n_degenerate = int(is_degenerate.sum())
        if n_degenerate:
            cols_to_show = [c for c in ['symbol', 'layer', 'horizon', 'model', 'split'] if c in out.columns]
            print(f'Dropping {n_degenerate} degenerate/overfit row(s) (accuracy=1 & f1_macro=0, or rmse=0 & DA=1):')
            display(out.loc[is_degenerate, cols_to_show].drop_duplicates().sort_values(cols_to_show))
        out = out[~is_degenerate].reset_index(drop=True)

    return out


leaderboard_long = load_all_leaderboards()

print('Rows:', len(leaderboard_long))
print('Models:', leaderboard_long['model'].nunique() if not leaderboard_long.empty else 0)
print('Layers:', leaderboard_long['layer'].dropna().unique().tolist() if not leaderboard_long.empty else [])
print('Horizons:', sorted(leaderboard_long['horizon'].dropna().unique().tolist()) if not leaderboard_long.empty else [])

display(leaderboard_long.head())


In [ ]:
leaderboard_long.head()

### New: degenerate / overfit-row filter (Section 3)

Added a hard filter, applied to **every** split (train / val / train_val / test) right
where `EXCLUDED_MODELS` is already dropped: any row with `accuracy == 1` **and**
`f1_macro == 0` (classification), or `rmse == 0` **and** `directional_acc == 1`
(regression), is removed before anything else runs. Both combinations are
mathematically pathological — a "perfect" model with zero macro-F1, or zero error with
perfect directional accuracy — and are a reliable signature of a broken evaluation row
(e.g. a class absent from a TEST fold) rather than a genuinely good model. The dropped
rows are printed in full so you can trace them back to the source `leaderboard.csv` if
you want to fix the upstream evaluation instead of just filtering it here.

See also **Section 10b** below for a broader, non-binary overfitting check (train vs.
test generalization gap) — this filter only catches the exact degenerate signature; a
model can still overfit without producing it.

## 4. Data-quality checks

These checks are intentionally diagnostic. Missing horizons/models are **not silently filled**.

In [ ]:
if leaderboard_long.empty:
    print('No leaderboard files were loaded. Check GDRIVE_BASE and the artifact paths.')
else:
    print('Columns available:')
    print(leaderboard_long.columns.tolist())

    print('\nRows by layer / horizon / split:')
    display(leaderboard_long.groupby(['layer','horizon','split']).size().rename('rows').reset_index().sort_values(['layer','horizon','split']).head())

    print('\nModels by layer:')
    display(leaderboard_long.groupby('layer')['model'].nunique().rename('n_models').reset_index().head())

    print('\nDuplicate model/split records:')
    dup_keys = ['symbol','layer','horizon','model','split']
    dup = leaderboard_long[leaderboard_long.duplicated(dup_keys, keep=False)]
    display(dup.sort_values(dup_keys).head() if not dup.empty else pd.DataFrame({'status':['No duplicates']}).head())

## 5. Test-set leaderboard — long format

For model comparison, the default leaderboard is the **TEST split only**, because the Phase-05 protocol refits on TRAIN+VAL and evaluates once on TEST.

In [ ]:
test_leaderboard_long = leaderboard_long[leaderboard_long['split'].astype(str).str.lower().eq('test')].copy()

# test_leaderboard_long = test_leaderboard_long.sort_values(['symbol','layer','horizon','model']).reset_index(drop=True)

print(f'TEST leaderboard rows: {len(test_leaderboard_long):,}')
display(test_leaderboard_long.head())

## 6. Publication-style tables — one per output variable (Symbol × Model × Horizon)

For **every output variable / layer** in `LAYER_CONFIG`, this builds a single wide DataFrame shaped like:

| Symbol | Model | Horizon=1 &nbsp;Metric A | Horizon=1 &nbsp;Metric B | Horizon=3 &nbsp;Metric A | ... |
|---|---|---:|---:|---:|---|

The metrics used depend on the layer's `task`:

- **Classification** (`L1_1_TRI`, `L1_1_DIR`): **Accuracy** and **F1 Macro**.
- **Regression** (`L1_2_*`, `L1_3_*`): **RMSE** and **Directional Accuracy**.

Only TEST-split rows are used, and the row index includes **Symbol** so multiple symbols (e.g. NIFTY, BANKNIFTY, NIFTYNEXT50) can be compared side by side in the same table.

In [ ]:
# Human-readable labels for the metrics that can appear in the publication tables.
METRIC_LABELS = {
    'accuracy': 'Accuracy',
    'f1_macro': 'F1 Macro',
    'rmse': 'RMSE',
    'directional_acc': 'Direction Accuracy',
}


def publication_table(df, layer, symbols=SYMBOLS, horizons=HORIZONS, metrics=None):
    """Build a wide Symbol x Model-Type x Model leaderboard for a single output
    variable (layer).

    Columns are a MultiIndex: (Horizon label, Metric label).
    Rows are a MultiIndex: (Symbol, Model Type, Model) — the "Model Type" level
    (Machine Learning Models / Deep Learning Models / Hybrid and Ensemble
    Models) is what makes the category headers appear, matching the reference
    reporting table.
    Metrics default to LAYER_CONFIG[layer]['primary_metrics']:
        classification -> ['accuracy', 'f1_macro']
        regression     -> ['rmse', 'directional_acc']
    """
    if metrics is None:
        metrics = LAYER_CONFIG.get(layer, {}).get('primary_metrics', [])

    x = df[df['layer'].eq(layer)].copy()
    if symbols is not None:
        x = x[x['symbol'].isin(symbols)]
    x = x[x['split'].astype(str).str.lower().eq('test')]

    if x.empty:
        return pd.DataFrame()

    available_h = [h for h in horizons if h in set(x['horizon'].dropna().astype(int))]

    rows = []
    for (symbol, model), g in x.groupby(['symbol', 'model'], sort=False):
        row = {
            'Symbol': symbol,
            'Model Type': MODEL_TO_TYPE.get(model, 'Other'),
            'Model': model,
        }
        for h in available_h:
            gh = g[g['horizon'].astype(int).eq(h)]
            for metric in metrics:
                val = gh[metric].iloc[0] if (not gh.empty and metric in gh.columns) else np.nan
                label = METRIC_LABELS.get(metric, metric)
                row[(f'Horizon = {h}', label)] = val
        rows.append(row)

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    metric_cols = [c for c in out.columns if c not in ('Symbol', 'Model Type', 'Model')]

    # Keep symbols in the order given. Within each symbol, order models by
    # category (Machine Learning -> Deep Learning -> Hybrid and Ensemble, per
    # MODEL_ORDER); any model not found in MODEL_ORDER is appended
    # alphabetically at the end, under a 'Other' Model Type.
    symbol_order = [s for s in symbols if s in out['Symbol'].unique()]

    out['_symbol_rank'] = out['Symbol'].apply(lambda s: symbol_order.index(s) if s in symbol_order else len(symbol_order))
    out['_model_rank'] = out['Model'].apply(lambda m: MODEL_RANK.get(m, len(MODEL_ORDER)))
    out = out.sort_values(['_symbol_rank', '_model_rank', 'Model'], kind='stable').drop(columns=['_symbol_rank', '_model_rank'])

    out = out.set_index(['Symbol', 'Model Type', 'Model'])
    out.columns = pd.MultiIndex.from_tuples([tuple(c) for c in metric_cols])
    return out

In [ ]:
# Build one publication table per output variable (layer), for every symbol at once.
publication_tables = {}

for layer in LAYER_CONFIG:
    table = publication_table(test_leaderboard_long, layer, symbols=SYMBOLS, horizons=HORIZONS)
    publication_tables[layer] = table

    label = LAYER_CONFIG[layer]['label']
    task = LAYER_CONFIG[layer]['task']
    print(f"--- {layer} ({label}) — task: {task} ---")
    if table.empty:
        print('  No TEST rows found for this layer/symbol combination yet.\n')
    else:
        display(table)
        print()

## 7. Convenience accessors

`publication_tables` is a dict keyed by layer name (e.g. `L1_1_TRI`, `L1_2_RVOL`). Each value is the Symbol × Model × Horizon table built above — this replaces the separate per-layer classification/regression helpers used previously.

In [ ]:
L1_1_TRI_test_df = publication_tables.get('L1_1_TRI', pd.DataFrame())

print('L1_1_TRI — Ternary regime detection')
display(L1_1_TRI_test_df)

## 8. Rank models at each horizon

This creates a compact ranking DataFrame from TEST results, per symbol. For classification, higher Accuracy/F1 is better. For regression, RMSE is lower-is-better while Directional Accuracy is higher-is-better.

In [ ]:
def rank_test_models(df, layer, metric, symbol=None, horizons=HORIZONS, ascending=False):
    x = df[(df['layer'] == layer) & (df['split'].astype(str).str.lower() == 'test')].copy()
    if symbol is not None:
        x = x[x['symbol'] == symbol]
    if metric not in x.columns or x.empty:
        return pd.DataFrame()

    x = x[x['horizon'].isin(horizons)].copy()
    x['rank'] = x.groupby('horizon')[metric].rank(method='min', ascending=ascending, na_option='bottom')
    return x[['symbol', 'layer', 'horizon', 'model', 'n', metric, 'rank']].sort_values(['horizon', 'rank', 'model'])


if not test_leaderboard_long.empty:
    tri_accuracy_ranking = rank_test_models(
        test_leaderboard_long, 'L1_1_TRI', 'accuracy', symbol=SYMBOLS[0], ascending=False
    )
    tri_f1_ranking = rank_test_models(
        test_leaderboard_long, 'L1_1_TRI', 'f1_macro', symbol=SYMBOLS[0], ascending=False
    )
    display(tri_accuracy_ranking)
    display(tri_f1_ranking)
else:
    print('No TEST results available.')

## 9. Best model by horizon — generalized across all three symbols

Section 9 above ranks models **within a single symbol**. For reporting a single "winning"
configuration per output variable × horizon, use instead:

> **Best model(layer, horizon) = the model with the best *average* TEST performance across
> NIFTY50, BANKNIFTY, and NIFTYNEXT50.**

Why average across symbols rather than, e.g., picking the single best-case symbol or taking a
majority vote:

- A model that only performs well on one index is a **less robust** candidate for a general
  "regime-aware forecasting" recommendation than one that performs consistently across all
  three market segments — this is exactly what H5 / RQ5 in the report are asking about (does
  a model's/signal's usefulness differ by segment?).
- Averaging keeps a single-symbol outlier from deciding the winner, and is directly comparable
  to how the report's Tables 4.1–4.7 already list one row per model spanning all three symbols.
- A model only "wins" a horizon if it has a TEST result for **every** symbol in `SYMBOLS` at
  that horizon (`n_symbols == len(SYMBOLS)`); pass a smaller `min_symbols` to
  `best_model_by_layer_horizon` if partial coverage should still be allowed to compete.

**Ranking metric per task (updated):**

- **Classification** (`L1_1_TRI`, `L1_1_DIR`): ranked by mean **Accuracy** across symbols —
  the primary, most directly interpretable metric. Mean **Macro-F1** is still computed and
  carried alongside on every table/plot, since Accuracy alone can be misleading under class
  imbalance (e.g. the ternary regime target) — F1 is the metric to check *when the two
  disagree* about how good a "win" really is.
- **Regression** (`L1_2_*`, `L1_3_*`): ranked by mean **RMSE** across symbols (lower is
  better); mean Directional Accuracy is reported alongside, unchanged.

The result, `best_model_summary`, has one row per `(layer, horizon)` — the natural answer to
"what's the overall best model for the return-magnitude forecast at horizon 10?", etc.

In [ ]:
# ------------------------------------------------------------------
# Generalize "best model" across all three symbols, for every layer/horizon.
#
#   Best model(layer, horizon) = the model with the best MEAN test
#   performance across NIFTY50, BANKNIFTY, and NIFTYNEXT50.
#
# See the markdown cell above for why averaging (rather than a per-symbol
# winner or a majority vote) is the right generalization here.
# ------------------------------------------------------------------

def aggregate_across_symbols(df, layer, metrics, symbols=SYMBOLS, horizons=HORIZONS, min_symbols=None):
    """Average TEST metrics for one layer, across symbols, per (horizon, model).

    Only (horizon, model) combinations observed for at least `min_symbols`
    distinct symbols are kept, so the comparison stays apples-to-apples --
    a model that only has results on one index cannot "win" by default.
    Defaults to requiring every symbol in `symbols`.
    """
    if min_symbols is None:
        min_symbols = len(symbols)

    x = df[(df['layer'] == layer) & (df['split'].astype(str).str.lower() == 'test')].copy()
    x = x[x['symbol'].isin(symbols) & x['horizon'].isin(horizons)]
    if x.empty:
        return pd.DataFrame()

    agg_kwargs = {'n_symbols': ('symbol', 'nunique')}
    for m in metrics:
        if m in x.columns:
            agg_kwargs[f'mean_{m}'] = (m, 'mean')
            agg_kwargs[f'std_{m}'] = (m, 'std')

    grouped = x.groupby(['horizon', 'model'], as_index=False).agg(**agg_kwargs)
    grouped = grouped[grouped['n_symbols'] >= min_symbols].copy()
    grouped.insert(0, 'layer', layer)
    return grouped.sort_values(['horizon', 'model']).reset_index(drop=True)


def best_model_by_layer_horizon(df, layer, cfg, symbols=SYMBOLS, horizons=HORIZONS, min_symbols=None):
    """For each horizon, pick the single model with the best across-symbol
    average performance for this layer.

    Ranking metric per task (updated):
      - classification -> mean Accuracy   (higher is better; primary metric)
      - regression      -> mean RMSE       (lower is better; primary metric)
    The secondary metric (Macro-F1 / Directional Accuracy) is carried along
    for reference but does not affect the ranking.
    """
    task = cfg['task']
    primary_metric, secondary_metric = cfg['primary_metrics']
    rank_metric = primary_metric  # Accuracy for classification, RMSE for regression.
    higher_is_better = (task == 'classification')

    metrics_needed = list(dict.fromkeys([primary_metric, secondary_metric, rank_metric]))
    agg = aggregate_across_symbols(df, layer, metrics_needed, symbols, horizons, min_symbols)
    if agg.empty:
        return pd.DataFrame()

    rank_col = f'mean_{rank_metric}'
    if rank_col not in agg.columns:
        return pd.DataFrame()

    idx = (agg.groupby('horizon')[rank_col].idxmax() if higher_is_better
           else agg.groupby('horizon')[rank_col].idxmin())
    best = agg.loc[idx].sort_values('horizon').reset_index(drop=True)
    best.insert(1, 'label', cfg['label'])
    best.insert(2, 'task', task)
    best['ranking_metric'] = rank_metric
    best['higher_is_better'] = higher_is_better
    return best


best_model_summary_frames = []
for layer, cfg in LAYER_CONFIG.items():
    best = best_model_by_layer_horizon(test_leaderboard_long, layer, cfg)
    if best.empty:
        print(f"--- {layer} ({cfg['label']}) — no horizon has results for all {len(SYMBOLS)} symbols yet ---\n")
        continue
    best_model_summary_frames.append(best)
    print(f"--- {layer} ({cfg['label']}) — best model by horizon (avg across {len(SYMBOLS)} symbols) ---")
    display(best)
    print()

best_model_summary = (
    pd.concat(best_model_summary_frames, ignore_index=True)
    if best_model_summary_frames else pd.DataFrame()
)

print('Combined best-model summary (all layers, all horizons):')
display(best_model_summary)


## 10. Save aggregated DataFrames

Outputs are written to a separate `Artifacts/results/leaderboard_aggregated/`
folder so the original per-run/per-horizon CSVs are never overwritten. One
publication CSV is saved per output variable (layer), plus the generalized
`best_model_summary` table from Section 9b.


In [ ]:
AGG_ROOT = RESULTS_ROOT / 'leaderboard_aggregated'
AGG_ROOT.mkdir(parents=True, exist_ok=True)

symbols_tag = '_'.join(SYMBOLS)

# Raw consolidated long tables.
leaderboard_long.to_csv(AGG_ROOT / f'{symbols_tag}_leaderboard_all_raw.csv', index=False)
test_leaderboard_long.to_csv(AGG_ROOT / f'{symbols_tag}_leaderboard_test_long.csv', index=False)

# Publication-style table for every output variable (layer).
for layer, table in publication_tables.items():
    if not table.empty:
        table.to_csv(AGG_ROOT / f'{symbols_tag}_{layer}_test_publication.csv')

# Generalized "best model per (layer, horizon), averaged across symbols" table (Section 9b).
if not best_model_summary.empty:
    best_model_summary.to_csv(AGG_ROOT / f'{symbols_tag}_best_model_summary.csv', index=False)

print('Saved aggregated outputs to:')
print(AGG_ROOT)


### 10b. Generalization gap check — TRAIN_VAL vs. TEST (new)

The degenerate-row filter in Section 3 only catches one exact, pathological signature.
A model can still be **overfit** in a more ordinary sense — fitting TRAIN_VAL much better
than it generalizes to TEST — without ever producing that signature. This check makes that
visible directly: for every model, it computes the gap between TRAIN_VAL and TEST on the
primary metric (Accuracy for classification, RMSE for regression, shown as a *relative*
gap so it's comparable across the different regression targets), and plots the 10 models
with the **largest average gap** per task.

**How to read it.** A large positive bar means that model's TRAIN_VAL performance is much
better than its TEST performance — a classic overfitting red flag, and a reason to treat
that model's leaderboard "win" (if it has one) with extra scrutiny even though it survived
the hard filter above. This is a diagnostic, not an automatic filter — no rows are dropped
here, since a genuinely legitimate gap (e.g. from a small TEST fold) shouldn't be silently
removed the way the exact degenerate signature is.

In [ ]:
# ------------------------------------------------------------------
# 10b. Generalization gap: TRAIN_VAL vs TEST, primary metric per task.
# Classification uses the raw (0-1 scale) gap; regression uses a
# *relative* gap (as a fraction of the TEST value) so RMSE-scale
# differences across targets (RVOL vs MAXDRAW vs RET, ...) don't distort
# the ranking.
# ------------------------------------------------------------------

def generalization_gap_for_layer(df, layer, cfg, symbols=SYMBOLS, horizons=HORIZONS):
    task = cfg['task']
    primary_metric = cfg['primary_metrics'][0]

    x = df[(df['layer'] == layer) & df['symbol'].isin(symbols) & df['horizon'].isin(horizons)].copy()
    if x.empty or primary_metric not in x.columns:
        return pd.DataFrame()

    x['split'] = x['split'].astype(str).str.lower()
    x = x[x['split'].isin(['train_val', 'test'])]
    if x.empty:
        return pd.DataFrame()

    piv = x.pivot_table(index=['symbol', 'horizon', 'model'], columns='split', values=primary_metric).reset_index()
    if 'train_val' not in piv.columns or 'test' not in piv.columns:
        return pd.DataFrame()
    piv = piv.dropna(subset=['train_val', 'test']).copy()
    if piv.empty:
        return pd.DataFrame()

    if task == 'classification':
        piv['gap'] = piv['train_val'] - piv['test']  # 0-1 scale, positive = overfit
    else:
        # Relative gap: how much worse TEST rmse is than TRAIN_VAL rmse, as a
        # fraction of TEST rmse -- comparable across differently-scaled targets.
        piv['gap'] = np.where(piv['test'] != 0, (piv['test'] - piv['train_val']) / piv['test'].abs(), np.nan)

    piv['layer'] = layer
    piv['task'] = task
    piv['metric'] = primary_metric
    return piv.dropna(subset=['gap'])


gap_frames = []
for layer, cfg in LAYER_CONFIG.items():
    g = generalization_gap_for_layer(leaderboard_long, layer, cfg)
    if not g.empty:
        gap_frames.append(g)

gap_long = pd.concat(gap_frames, ignore_index=True) if gap_frames else pd.DataFrame()

if gap_long.empty:
    print("No TRAIN_VAL / TEST pairs available yet -- skipping the generalization-gap check.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    for ax, task_name in zip(axes, ["classification", "regression"]):
        sub = gap_long[gap_long["task"] == task_name]
        if sub.empty:
            ax.set_visible(False)
            continue

        summary = (
            sub.groupby('model', as_index=False)
            .agg(mean_gap=('gap', 'mean'), std_gap=('gap', 'std'), n_obs=('gap', 'size'))
            .sort_values('mean_gap', ascending=False)
            .head(10)
            .iloc[::-1]
        )
        colors = [color_for_model(m) for m in summary['model']]

        ax.barh(summary['model'], summary['mean_gap'], xerr=summary['std_gap'].fillna(0),
                color=colors, edgecolor='black', linewidth=0.4)
        ax.axvline(0, color='black', linewidth=0.8)

        xlabel = (
            "Mean TRAIN_VAL - TEST Accuracy gap\n(higher = more overfit)" if task_name == "classification"
            else "Mean relative RMSE gap\n(higher = TRAIN_VAL fits much better than TEST)"
        )
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_title(task_name.capitalize(), fontsize=12)
        ax.grid(axis='x', alpha=0.3)

    fig.legend(handles=category_legend_handles(), loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05))
    fig.suptitle('Top 10 Models by TRAIN_VAL -> TEST Generalization Gap (Primary Metric)', fontsize=14, y=1.03)

    fig.tight_layout(rect=[0, 0.05, 1, 1])
    save_fig(fig, AGG_ROOT, 'generalization_gap_top_offenders')
    plt.show()

    print(
        "Suggested caption: \"Models with the largest average TRAIN_VAL -> TEST gap on the "
        "primary metric per task (Accuracy for classification, relative RMSE gap for "
        "regression), averaged across symbols and horizons. A large positive gap flags "
        "models whose TEST 'win' should be treated with extra scrutiny.\"\n"
    )


# 11. Generalized Best-Model Visualizations

This section visualizes the generalized TEST model-selection results.

The selected model for each `(layer, horizon)` is the model with the best
average TEST performance across NIFTY50, BANKNIFTY, and NIFTYNEXT50 --
already computed as `plot_df` (a copy of `best_model_summary`) in 11.1, so
every plot below shows the single **Top-1** winner per target/horizon,
not a shortlist.

For classification, **Accuracy** is the ranking metric (Macro-F1 shown
alongside on every plot, since Accuracy alone can be misleading under
class imbalance).

For regression, **RMSE** is the ranking metric (Directional Accuracy shown
alongside).

The forest/dot-and-whisker plots show the mean performance across the
three indices together with the cross-index standard deviation.

In [ ]:
# ============================================================
# 11.1 Prepare generalized best-model plotting data
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Copy the generalized best-model summary created in Cell 22
# ------------------------------------------------------------

plot_df = best_model_summary.copy()

if plot_df.empty:
    raise ValueError(
        "best_model_summary is empty. Run Section 9b / Cell 22 first."
    )

# ------------------------------------------------------------
# Ensure horizon is numeric
# ------------------------------------------------------------

plot_df["horizon"] = pd.to_numeric(
    plot_df["horizon"],
    errors="coerce"
)

# ------------------------------------------------------------
# Human-readable labels
# ------------------------------------------------------------

TARGET_LABELS = {
    "L1_1_DIR": "Binary Direction",
    "L1_1_TRI": "Trinary Regime",
    "L1_2_RVOL": "Realised Volatility",
    "L1_2_MAXDRAW": "Maximum Drawdown",
    "L1_2_MAXRET": "Maximum Forward Return",
    "L1_3_LOG": "Forward Log Return",   # kept in case this key is produced elsewhere
    "L1_3_PCT": "Forward Percentage Return",
    "L1_3_RET": "Forward Return",        # FIX: was missing -- L1_3_RET (a real layer in
    # LAYER_CONFIG) used to fall through to the raw cfg['label'] string instead, which
    # produced ugly plot titles/filenames like 'layer_1.3_—_direction_&_return_forecasting'.
}

plot_df["target_label"] = (
    plot_df["layer"]
    .map(TARGET_LABELS)
    .fillna(plot_df["label"])
)

# ------------------------------------------------------------
# Make task labels consistent
# ------------------------------------------------------------

plot_df["task"] = (
    plot_df["task"]
    .astype(str)
    .str.lower()
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    plot_df[
        [
            "layer",
            "target_label",
            "horizon",
            "model",
            "task",
            "ranking_metric",
            "n_symbols"
        ]
    ].sort_values(
        ["layer", "horizon"]
    )
)

### 11.3 Top-1 model per target, by horizon — forest / dot-and-whisker plots (updated: Top-1, Accuracy-ranked)

**What changed.** This used to show the top **3** models per horizon, ranked by Macro-F1.
It now shows only the single **winning** model per horizon (matching `plot_df` /
`best_model_summary` directly — no separate Top-3 pipeline needed any more), ranked by
**Accuracy** for classification and **RMSE** for regression. The secondary metric
(Macro-F1 / Directional Accuracy) is printed next to each point as an annotation, so you
still see both numbers without needing three rows per horizon.

**What it shows.** One row per horizon — the dot is the **mean** primary-metric score
across the three indices, the whisker is the **standard deviation** across those three
indices (i.e. symbol-to-symbol consistency, not a confidence interval). The annotation to
the right of each dot gives the secondary metric's mean.

**How to read it.** Dot color = model category (see legend). A short whisker means the
winner is consistent across NIFTY50 / BANKNIFTY / NIFTYNEXT50; a long whisker means the
"win" is being driven by strong performance on one or two indices rather than all three.
If Accuracy and the annotated F1/DA disagree sharply for a horizon (e.g. high accuracy but
noticeably lower F1), that is worth flagging in the paper as a sign of class imbalance
rather than genuinely strong classification.

In [ ]:
# ============================================================
# 11.3 Top-1 model per target, by horizon  (publication style)
#
# One dot per horizon = the single generalized winner (mean TEST
# performance across NIFTY50 + BANKNIFTY + NIFTYNEXT50), taken directly
# from `plot_df` (Section 11.1) -- no separate Top-3 pipeline needed.
#
# Encoding:
#   - x position + whisker -> primary metric (Accuracy / RMSE), mean +/- SD
#     across the three indices.
#   - dot color             -> model category (see legend).
#   - text annotation       -> secondary metric (F1-macro / Directional Acc).
# ============================================================

if plot_df.empty:
    print("plot_df is empty -- run Section 11.1 first.")
else:
    for target_label in plot_df["target_label"].unique():
        sub = plot_df[plot_df["target_label"] == target_label].sort_values("horizon").reset_index(drop=True)
        if sub.empty:
            continue

        task = sub["task"].iloc[0]
        if task == "classification":
            primary_col, primary_std_col, primary_label = "mean_accuracy", "std_accuracy", "Accuracy"
            secondary_col, secondary_label = "mean_f1_macro", "F1-macro"
        else:
            primary_col, primary_std_col, primary_label = "mean_rmse", "std_rmse", "RMSE"
            secondary_col, secondary_label = "mean_directional_acc", "Dir. Acc."

        sub = sub.iloc[::-1].reset_index(drop=True)  # h=1 at top
        y = np.arange(len(sub))
        colors = [color_for_model(m) for m in sub["model"]]

        fig, ax = plt.subplots(figsize=(12, max(3.2, len(sub) * 0.6)))

        ax.errorbar(
            sub[primary_col], y, xerr=sub[primary_std_col].fillna(0),
            fmt="none", ecolor="0.6", elinewidth=1.3, capsize=4, zorder=2,
        )
        ax.scatter(sub[primary_col], y, c=colors, s=100, edgecolor="black", linewidth=0.7, zorder=3)

        for i in range(len(sub)):
            xv, sv = sub.loc[i, primary_col], sub.loc[i, secondary_col]
            if pd.notna(xv) and pd.notna(sv):
                ax.annotate(
                    f"{secondary_label} = {sv:.3f}", (xv, i), xytext=(18, 0),
                    textcoords="offset points", va="center", fontsize=8.5, color="0.25",
                    bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85),
                )

        ax.set_yticks(y)
        ax.set_yticklabels([f"h={int(h)}   {m}" for h, m in zip(sub["horizon"], sub["model"])], fontsize=10)
        ax.set_xlabel(f"Mean TEST {primary_label} across three indices", fontsize=11)
        ax.set_title(f"Top Model: {target_label}", fontsize=11, pad=12)

        xmin, xmax = ax.get_xlim()
        ax.set_xlim(xmin, xmax + 0.40 * (xmax - xmin))  # room for the annotations
        ax.legend(
            handles=category_legend_handles(), title="Model category",
            loc="center left", bbox_to_anchor=(1.01, 0.5),
        )
        ax.grid(axis="x", alpha=0.30)

        filename = "top1_" + target_label.lower().replace(" ", "_").replace("-", "")
        fig.tight_layout()
        save_fig(fig, AGG_ROOT, filename)
        plt.show()

        print(
            f"Suggested caption: \"Mean TEST {primary_label} (\u00b1 cross-index SD) of the "
            f"winning model for {target_label}, by forecast horizon; mean TEST {secondary_label} "
            f"shown alongside each point. Marker color denotes model category.\"\n"
        )


### 11.4 Best-model heatmap (updated: now colored by model category)

**What changed.** The background used to be plain binary shading (did this cell have a
winner or not) — with 35/35 cells filled, that shading carried no information. It now
encodes the **category of the winning model** (blue / orange / green for ML / DL /
Hybrid), so the heatmap does two jobs at once: it names the winning model *and* shows,
visually, whether classical ML, deep learning, or ensembling dominates a given
target/horizon region.

**How to read it.** Look for color *bands*: e.g. if a whole row is blue, that target is
consistently won by classical ML regardless of horizon; if color shifts from blue to
orange as horizon increases, that's evidence deep learning becomes relatively more
competitive at longer horizons for that target — a natural discussion point for the
paper's RQ on horizon effects.

**Note:** the annotated model name is unchanged; only the shading now carries extra
information instead of being purely decorative.

In [ ]:
heatmap_models

In [ ]:
TARGET_LABELS.keys()

In [ ]:
heatmap_models = heatmap_models.reindex(TARGET_LABELS.keys())
heatmap_models.index = heatmap_models.index.map(TARGET_LABELS.values())
heatmap_models

In [ ]:
# ============================================================
# 11.4 Best-model heatmap  (publication style)
#
# Cell color now encodes the WINNING MODEL'S CATEGORY (not just
# "present / missing"), so the heatmap answers "which family of model
# tends to win where?" at a glance, in addition to naming the model.
# ============================================================

heatmap_source = plot_df.copy()

heatmap_models = heatmap_source.pivot_table(
    index="target_label",
    columns="horizon",
    values="model",
    aggfunc="first"
)

# Desired row order
row_order = [
    "Binary Direction",
    "Trinary Regime",
    "Realised Volatility",
    "Maximum Drawdown",
    "Maximum Forward Return",
    "Forward Percentage Return",
    "Forward Return",
]

# Arrange rows
heatmap_models = heatmap_models.reindex(row_order)

horizon_order = [1, 3, 5, 10, 21]
horizon_available = [h for h in horizon_order if h in heatmap_models.columns]
heatmap_models = heatmap_models[horizon_available]

_cat_code = {'Machine Learning Models': 0, 'Deep Learning Models': 1, 'Hybrid and Ensemble Models': 2, 'Other': 3}


def _code_for(model):
    if pd.isna(model):
        return np.nan
    return _cat_code.get(MODEL_TO_TYPE.get(model, 'Other'), 3)


heatmap_codes = heatmap_models.apply(lambda col: col.map(_code_for))

from matplotlib.colors import ListedColormap

cmap = ListedColormap([
    CATEGORY_COLORS['Machine Learning Models'],
    CATEGORY_COLORS['Deep Learning Models'],
    CATEGORY_COLORS['Hybrid and Ensemble Models'],
    CATEGORY_COLORS['Other'],
])

fig, ax = plt.subplots(figsize=(11, 6.5))

im = ax.imshow(heatmap_codes.values.astype(float), aspect="auto", cmap=cmap, vmin=-0.5, vmax=3.5)

for i in range(heatmap_models.shape[0]):
    for j in range(heatmap_models.shape[1]):
        value = heatmap_models.iloc[i, j]
        if pd.notna(value):
            ax.text(j, i, str(value), ha="center", va="center", fontsize=8, color="white")

ax.set_xticks(range(len(heatmap_models.columns)))
ax.set_xticklabels([f"h={h}" for h in heatmap_models.columns], fontsize=10)
ax.set_yticks(range(len(heatmap_models.index)))
ax.set_yticklabels(heatmap_models.index, fontsize=9)
ax.set_xlabel("Forecast Horizon (Trading Days)", fontsize=11)
ax.set_ylabel("Forecasting Target", fontsize=11)
ax.set_title("Best-Performing Model Across Targets and Forecast Horizons", fontsize=11, pad=15)

ax.set_xticks(np.arange(-0.5, len(heatmap_models.columns), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(heatmap_models.index), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1.2)
ax.grid(which="major", visible=False)
ax.tick_params(which="minor", bottom=False, left=False)

ax.legend(
    handles=category_legend_handles(), loc="upper center",
    bbox_to_anchor=(0.5, -0.14), ncol=3, title="Model category",
)

fig.tight_layout()
save_fig(fig, AGG_ROOT, "best_model_heatmap")
plt.show()

print(
    "Suggested caption: \"Best-performing model for each forecasting target and horizon "
    "(mean TEST performance across NIFTY50, BANKNIFTY and NIFTYNEXT50). Cell color denotes "
    "the winning model's category.\"\n"
)


### 11.5 Model-selection frequency (updated: bars now colored by category)

**What it shows.** How many of the 35 (target × horizon) "winner slots" each model
architecture captured. **What changed:** bars are now colored by model category with the
same legend used everywhere else in the notebook, so this chart visually agrees with the
heatmap above it (e.g. a model with a tall orange bar here should correspond to orange
cells in Section 11.4).

**Caveat worth keeping in the paper.** This count treats every (target, horizon) slot as
equally important — winning at h=21 on Realised Volatility counts the same as winning at
h=1 on Trinary Regime. If some targets/horizons matter more for the paper's narrative,
say so explicitly rather than reading this chart as an unweighted "best overall model"
ranking.

In [ ]:
# ============================================================
# 11.5 Model-selection frequency  (publication style)
#
# Each bar is now colored by the model's category, with a shared legend,
# consistent with every other model-colored plot in the notebook.
# ============================================================

frequency_df = (
    plot_df["model"]
    .value_counts()
    .reset_index()
)

frequency_df.columns = ["model", "selection_count"]
frequency_df = frequency_df.sort_values("selection_count", ascending=False)

display(frequency_df)

bar_colors = [color_for_model(m) for m in frequency_df["model"]]

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.bar(
    frequency_df["model"], frequency_df["selection_count"],
    color=bar_colors, edgecolor="black", linewidth=0.5,
)

ax.set_xlabel("Model Architecture", fontsize=11)
ax.set_ylabel("Number of Target-Horizon Selections", fontsize=11)
ax.set_title("Frequency of Model Selection Across Targets and Horizons", fontsize=13)

ax.tick_params(axis="x", rotation=45)
for lbl in ax.get_xticklabels():
    lbl.set_ha("right")

ax.grid(axis="y", alpha=0.3)
ax.legend(handles=category_legend_handles(), loc="upper right", title="Model category")

fig.tight_layout()
save_fig(fig, AGG_ROOT, "model_selection_frequency")
plt.show()

print(
    "Suggested caption: \"Number of (target, horizon) combinations for which each model "
    "was the generalized winner (mean TEST performance across all three indices). Bar "
    "color denotes model category.\"\n"
)


## 11.6 Alternative / consolidated best-model visualizations

Sections 11.3-11.5 above draw one full-size figure per target (7 separate
forest plots), plus a heatmap and a frequency chart. The cells below add a
couple of alternative ways to look at the *same* generalized best-model
results (`plot_df`), consolidated so everything fits in fewer figures:

- **11.6** -- all Top-1 forest plots as small multiples in a single figure.
- **11.8** -- slope/trend chart showing how the winning score moves across
  horizons for every target, split by task.

(A grouped-bar version of 11.8 was removed in this revision -- it showed
the same underlying numbers as the slope chart with more visual clutter
for 7 targets x 5 horizons; the slope chart is the cleaner read.)

### 11.6 Small-multiples version of 11.3 (updated: Top-1, category colors, one shared legend)

Same Top-1 winners as Section 11.3, compressed into a single figure — useful when the
paper's page budget only fits one figure instead of seven. Point color again encodes model
category (shared legend at the bottom).

In [ ]:
# ============================================================
# 11.6 Consolidated small-multiples forest plot  (Top-1, publication style)
#
# Same Top-1 winners as 11.3, one compact subplot per target.
# ============================================================

if plot_df.empty:
    print("plot_df is empty -- run Section 11.1 first.")
else:
    target_labels_all = plot_df["target_label"].unique().tolist()
    n_targets = len(target_labels_all)
    n_cols = 2
    n_rows = int(np.ceil(n_targets / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, max(4.5, n_rows * 2.6)), squeeze=False)

    for idx, target_label in enumerate(target_labels_all):
        ax = axes[idx // n_cols][idx % n_cols]

        sub = plot_df[plot_df["target_label"] == target_label].sort_values("horizon").reset_index(drop=True)
        task = sub["task"].iloc[0]
        if task == "classification":
            primary_col, primary_std_col, primary_label = "mean_accuracy", "std_accuracy", "Accuracy"
        else:
            primary_col, primary_std_col, primary_label = "mean_rmse", "std_rmse", "RMSE"

        sub = sub.iloc[::-1].reset_index(drop=True)
        y = np.arange(len(sub))
        colors = [color_for_model(m) for m in sub["model"]]

        ax.errorbar(
            sub[primary_col], y, xerr=sub[primary_std_col].fillna(0),
            fmt="none", ecolor="0.6", elinewidth=1.0, capsize=3, zorder=2,
        )
        ax.scatter(sub[primary_col], y, c=colors, s=55, edgecolor="black", linewidth=0.5, zorder=3)

        ax.set_yticks(y)
        ax.set_yticklabels([f"h={int(h)}  {m}" for h, m in zip(sub["horizon"], sub["model"])], fontsize=7)
        ax.set_title(target_label, fontsize=10)
        ax.set_xlabel(f"Mean TEST {primary_label}", fontsize=8)
        ax.grid(axis="x", alpha=0.30)

    for idx in range(n_targets, n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].axis("off")

    fig.legend(
        handles=category_legend_handles(), loc="lower center", ncol=3,
        bbox_to_anchor=(0.5, -0.02), title="Model category",
    )
    fig.suptitle("Top-1 Model by Target -- All Targets, One Figure", fontsize=14)

    fig.tight_layout(rect=[0, 0.04, 1, 1])
    save_fig(fig, AGG_ROOT, "forest_top1_all_targets_grid")
    plt.show()


### 11.8 Winning score trend across horizons (unchanged logic, new palette)

One line per target, so you can see at a glance which targets **degrade quickly** as the
forecast horizon lengthens (a steep downward slope for regression, or downward for
classification since higher F1 is better) versus which stay flat. This is usually one of
the more citable figures in the paper for discussing horizon effects, since it summarizes
all 5 horizons x 7 targets in one panel per task.

In [ ]:
# ============================================================
# 11.8 Slope / trend chart -- winning score across horizons
#
# One line per target, x = horizon, y = winning score. Useful for
# spotting targets whose best achievable score degrades quickly with
# horizon vs. targets that stay stable. Axis label is now derived from
# the actual ranking metric (Accuracy / RMSE) instead of being hardcoded.
# ============================================================

METRIC_AXIS_LABELS = {
    'accuracy': 'Mean TEST Accuracy',
    'f1_macro': 'Mean TEST Macro-F1',
    'rmse': 'Mean TEST RMSE',
    'directional_acc': 'Mean TEST Directional Accuracy',
}

if plot_df.empty:
    print("plot_df is empty -- run Section 11.1 (Cell 26) first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    for ax, task_name in zip(axes, ["classification", "regression"]):
        sub = plot_df[plot_df["task"] == task_name].copy()

        if sub.empty:
            ax.set_visible(False)
            continue

        rank_metric = sub['ranking_metric'].iloc[0]
        score_col = f"mean_{rank_metric}"
        ylabel = METRIC_AXIS_LABELS.get(rank_metric, score_col)

        for target_label, g in sub.groupby("target_label"):
            g = g.sort_values("horizon")
            ax.plot(g["horizon"], g[score_col], marker="o", label=target_label)

        ax.set_xlabel("Forecast Horizon (Trading Days)", fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(task_name.capitalize(), fontsize=12)
        ax.legend(fontsize=7, loc="best")
        ax.grid(alpha=0.30)

    fig.suptitle("Winning Model Score Trend Across Horizons", fontsize=14)

    fig.tight_layout()
    save_fig(fig, AGG_ROOT, "best_model_trend_by_horizon")
    plt.show()


## 11. Optional: exact screenshot-style table for LaTeX

The DataFrame can be converted directly to LaTeX. The following cell creates a clean table from any layer's publication table — change `latex_layer` to export a different output variable.

In [ ]:
# 'L1_1_DIR',
# 'L1_1_TRI',
# 'L1_2_MAXDRAW',
# 'L1_2_MAXRET',
# 'L1_2_RVOL',
# 'L1_3_PCT',
# 'L1_3_RET',

In [ ]:
latex_layer = 'L1_2_MAXDRAW'
latex_source = publication_tables.get(latex_layer, pd.DataFrame())

if not latex_source.empty:
    latex_table = latex_source.to_latex(
        na_rep='--',
        float_format=lambda x: f'{x:.3f}',
        escape=True,
        multirow=True,
    )
    print(latex_table)
else:
    print(f'{latex_layer} table is empty; run the loader after the relevant leaderboard files exist.')

### Notes

- The notebook uses **TEST rows only** for every publication leaderboard.
- `train`, `val`, and `train_val` remain available in `leaderboard_long` if you need overfitting/train-validation-gap analysis.
- Missing files/horizons/symbols are displayed as missing (`NaN`); they are not treated as zero.
- `publication_tables` contains one Symbol × Model × Horizon DataFrame **per output variable (layer)**, keyed by layer name:
  - Classification layers (`L1_1_TRI`, `L1_1_DIR`) use **Accuracy** and **F1 Macro**.
  - Regression layers (`L1_2_MAXDRAW`, `L1_2_MAXRET`, `L1_2_RVOL`, `L1_3_PCT`, `L1_3_RET`) use **RMSE** and **Direction Accuracy**.
- Set `SYMBOLS = ['NIFTY', 'BANKNIFTY', 'NIFTYNEXT50']` (or any subset) in the Configuration cell to reproduce the multi-symbol layout shown in the screenshot; each table stacks symbols as an outer row level above Model.

## 12. Feature Importance Aggregation — All Layers, Symbols & Horizons

This section reads the saved **`mi_shap_test.csv`** files produced by the Phase-05 model
notebooks and builds Symbol × Feature × Horizon (and Domain × Horizon) publication tables,
mirroring the leaderboard aggregation above.

Expected stable artifact structure:

```text
Artifacts/
└── feature_importance/
    └── {SYMBOL}/
        └── {LAYER}/
            └── h{HORIZON}d/
                ├── mi_shap_test.csv
                └── mi_shap_train.csv
```

e.g. `Artifacts/feature_importance/NIFTY50/L1_1_DIR/h1d/mi_shap_test.csv`.

Only the **TEST** file (`mi_shap_test.csv`) is used, matching the leaderboard convention of
reporting TEST-split results only; `mi_shap_train.csv` is not read.

Each `mi_shap_test.csv` has **one row per (target, feature)** — this is the *post-selection*
explainability output described in the paper (Section 3.4.4): a single train-only surrogate
model's SHAP values per target, not one row per candidate model, so there is nothing to
average across models here. The columns are:

| Column                  | Meaning                                                              |
|--------------------------|-----------------------------------------------------------------------|
| `symbol`                | NIFTY50 / BANKNIFTY / NIFTYNEXT50                                     |
| `layer`                 | Output-variable key, e.g. `L1_1_DIR`, `L1_2_RVOL`                     |
| `horizon`               | Forecast horizon in trading days (1, 3, 5, 10, 21)                    |
| `target`                | The specific forecasting target within the layer                      |
| `feature`               | Candidate feature name                                                |
| `mi_test`               | Mutual information between the feature and the target (test split)    |
| `shap_importance_test`  | Mean absolute SHAP value for the feature (test split)                 |
| `domain`                | Feature-engineering group the feature belongs to (Table 3.3), e.g.\
                            Price/Return, Momentum/Trend, Volatility/Volume, Macro/Geopolitical/\
                            Sentiment/Sustainability, Cross-Market, Nonlinear/Interaction, etc.  |

Two tables are produced below:

- **12.4** — a Symbol × Feature table (MI and SHAP per horizon), for inspecting individual
  top features, same layout as the leaderboard publication tables.
- **12.5b** — a **Domain × Horizon** table, generalized (averaged) across all three symbols,
  the feature-importance analogue of the Section 9b "best model" summary. This directly
  answers RQ4/RQ5: how does each information family's (market / macro / geopolitical /
  sentiment / sustainability / ...) *share* of predictive importance change across horizons
  and market segments?


In [ ]:
# ------------------------------------------------------------------
# Feature-importance artifact root & expected filename
# ------------------------------------------------------------------
FEATURE_IMPORTANCE_ROOT = ARTIFACT_ROOT / 'feature_importance'

# Only the TEST-split file is used (mirrors the leaderboard's TEST-only rule).
FEATURE_IMPORTANCE_FILENAME = 'mi_shap_test.csv'

print('FEATURE_IMPORTANCE_ROOT:', FEATURE_IMPORTANCE_ROOT)
print('FEATURE_IMPORTANCE_FILENAME:', FEATURE_IMPORTANCE_FILENAME)


### 12.1 Discover saved feature-importance files

Same discovery pattern as the leaderboard files: every combination of `SYMBOLS × LAYER_CONFIG × HORIZONS` is checked, and missing files are reported (not silently skipped).

In [ ]:
def discover_stable_feature_files(symbols=SYMBOLS, horizons=HORIZONS):
    rows = []
    for symbol in symbols:
        for layer in LAYER_CONFIG:
            for h in horizons:
                p = FEATURE_IMPORTANCE_ROOT / symbol / layer / f'h{h}d' / FEATURE_IMPORTANCE_FILENAME
                rows.append({
                    'symbol': symbol,
                    'layer': layer,
                    'horizon': h,
                    'path': str(p),
                    'exists': p.exists(),
                })
    return pd.DataFrame(rows)


feature_file_inventory = discover_stable_feature_files()
display(feature_file_inventory[feature_file_inventory['exists']].copy())

print(f'Found {feature_file_inventory["exists"].sum()} feature-importance (mi_shap_test) files.')
print(f'Missing {len(feature_file_inventory) - feature_file_inventory["exists"].sum()} expected files.')


### 12.2 Read and standardize all mi_shap_test.csv files

Reads each file as-is — one row per `(target, feature)`, with columns `symbol`, `layer`,
`horizon`, `target`, `feature`, `mi_test`, `shap_importance_test`, `domain` — and fills in any
missing `symbol` / `layer` / `horizon` metadata from the discovered file path (same pattern as
the leaderboard loader). No model-level averaging happens here, since the source file is not
keyed by model.


In [ ]:
# The exact columns present in every mi_shap_test.csv file.
FEATURE_IMPORTANCE_COLUMNS = [
    'symbol', 'layer', 'horizon', 'target', 'feature',
    'mi_test', 'shap_importance_test', 'domain',
]


def read_feature_importance_file(path, symbol=None, layer=None, horizon=None):
    df = pd.read_csv(path)

    # Prefer metadata already stored inside the CSV; otherwise use path metadata.
    for col, val in (('symbol', symbol), ('layer', layer), ('horizon', horizon)):
        if col not in df.columns or df[col].isna().all():
            df[col] = val

    missing = [c for c in FEATURE_IMPORTANCE_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f'{path} is missing expected column(s): {missing}')

    df = df[FEATURE_IMPORTANCE_COLUMNS].copy()
    df['source_file'] = str(path)
    return df


def load_all_feature_importance():
    frames = []

    stable = feature_file_inventory[feature_file_inventory['exists']].copy()
    for _, r in stable.iterrows():
        frames.append(read_feature_importance_file(
            r['path'], r['symbol'], r['layer'], int(r['horizon'])
        ))

    if not frames:
        return pd.DataFrame()

    out = pd.concat(frames, ignore_index=True, sort=False)
    out['horizon'] = pd.to_numeric(out['horizon'], errors='coerce').astype('Int64')
    out['mi_test'] = pd.to_numeric(out['mi_test'], errors='coerce')
    out['shap_importance_test'] = pd.to_numeric(out['shap_importance_test'], errors='coerce')
    out['domain'] = out['domain'].astype(str)

    return out


feature_importance_long = load_all_feature_importance()

print('Rows:', len(feature_importance_long))
if not feature_importance_long.empty:
    print('Layers:', feature_importance_long['layer'].dropna().unique().tolist())
    print('Horizons:', sorted(feature_importance_long['horizon'].dropna().unique().tolist()))
    print('Domains:', sorted(feature_importance_long['domain'].dropna().unique().tolist()))

display(feature_importance_long.head(20))


### 12.1b Global feature-domain color map (new)

The feature-importance plots in Sections 12 and 13 all color bars by **domain** (Momentum,
Volatility, Trend, ...). Previously each plotting cell built its own `tab10`-based color
map from whatever domains happened to appear in that cell's subset of the data — which
meant the same domain (e.g. "Momentum") could get a *different* color in two different
figures. The cell below fixes this by computing one color map from the **full** set of
domains, once, and every feature-importance plot from here on reuses it.

In [ ]:
DOMAIN_ORDER = sorted(feature_importance_long['domain'].dropna().unique()) if not feature_importance_long.empty else []
DOMAIN_COLORS = {d: OKABE_ITO[i % len(OKABE_ITO)] for i, d in enumerate(DOMAIN_ORDER)}


def domain_legend_handles(domains=None):
    domains = domains if domains is not None else DOMAIN_ORDER
    return [plt.Rectangle((0, 0), 1, 1, color=DOMAIN_COLORS[d], label=d) for d in domains if d in DOMAIN_COLORS]


print(f"{len(DOMAIN_ORDER)} feature domains found:", DOMAIN_ORDER)


### 12.3 De-duplicate to one row per (symbol, layer, horizon, target, feature)

`mi_test` and `shap_importance_test` are already target/feature-specific (not per-model), so
there is nothing to average across models. This step is only a safety net against accidental
duplicate rows across source files (e.g. if a file were re-saved) — a `n_dup_groups` check is
printed so you can confirm the source data really has one row per key.


In [ ]:
def resolve_feature_importance(df):
    """Collapse to exactly one row per (symbol, layer, horizon, target, feature).

    mi_test / shap_importance_test are computed once per target/feature (see
    Section 3.4.4 of the report -- a train-only surrogate model's SHAP
    values), so this is a safety net against accidental duplicate rows
    rather than a genuine "average across models" step.
    """
    if df.empty:
        return pd.DataFrame()

    key = ['symbol', 'layer', 'horizon', 'target', 'feature']
    dup_check = df.groupby(key).size()
    n_dup_groups = int((dup_check > 1).sum())
    if n_dup_groups:
        print(f'Note: {n_dup_groups} {tuple(key)} groups have duplicate rows -- using the mean.')

    out = df.groupby(key + ['domain'], as_index=False).agg(
        mi_test=('mi_test', 'mean'),
        shap_importance_test=('shap_importance_test', 'mean'),
    )
    return out


feature_importance_avg = resolve_feature_importance(feature_importance_long)
display(feature_importance_avg.head(20))


### 12.4 Publication-style table — one per layer (Symbol × Feature × Horizon × [MI, SHAP])

For **every layer** in `LAYER_CONFIG`, this builds a wide DataFrame with columns as a
MultiIndex `(Horizon = h, Metric)` — the same pattern used for the leaderboard publication
tables:

| Symbol | Features | Horizon = 1 MI | Horizon = 1 SHAP | Horizon = 3 MI | Horizon = 3 SHAP | ... |
|---|---|---:|---:|---:|---:|---|

Within each symbol, features are ordered by their **mean SHAP across all available horizons**
(most important first) by default; pass `sort_by='alphabetical'` or `sort_by='mi'` for a
different ordering. When a layer has more than one `target` (e.g. several regression targets
sharing a layer key), pass `target=...` to focus the table on a single target.


In [ ]:
FI_METRIC_LABELS = {'mi_test': 'MI', 'shap_importance_test': 'SHAP'}


def feature_importance_table(df, layer, symbols=SYMBOLS, horizons=HORIZONS, target=None,
                              metrics=('mi_test', 'shap_importance_test'), sort_by='shap_importance_test'):
    """Build a Symbol -> Feature table of MI / SHAP per horizon.

    sort_by:
        'shap_importance_test' -> within each symbol, rank features by mean SHAP
                                   across all available horizons (descending).
        'mi_test'               -> same, but ranked by mean MI instead.
        'alphabetical'          -> sort feature names alphabetically within each symbol.
    """
    x = df[df['layer'].eq(layer)].copy()
    if symbols is not None:
        x = x[x['symbol'].isin(symbols)]
    if target is not None:
        x = x[x['target'].eq(target)]
    if x.empty:
        return pd.DataFrame()

    available_h = [h for h in horizons if h in set(x['horizon'].dropna().astype(int))]

    rows = []
    for (symbol, feature), g in x.groupby(['symbol', 'feature'], sort=False):
        row = {'Symbol': symbol, 'Features': feature}
        for h in available_h:
            gh = g[g['horizon'].astype(int).eq(h)]
            for metric in metrics:
                val = gh[metric].iloc[0] if (not gh.empty and metric in gh.columns) else np.nan
                label = FI_METRIC_LABELS.get(metric, metric)
                row[(f'Horizon = {h}', label)] = val
        rows.append(row)

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    metric_cols = [c for c in out.columns if c not in ('Symbol', 'Features')]

    # Used only to order features within each symbol -- not shown in the output.
    sort_metric = 'mi_test' if sort_by == 'mi_test' else 'shap_importance_test'
    sort_cols = [(f'Horizon = {h}', FI_METRIC_LABELS[sort_metric]) for h in available_h]
    out['_overall'] = out[sort_cols].mean(axis=1, skipna=True)

    symbol_order = [s for s in symbols if s in out['Symbol'].unique()]
    out['_symbol_rank'] = out['Symbol'].apply(
        lambda s: symbol_order.index(s) if s in symbol_order else len(symbol_order)
    )

    if sort_by in ('shap_importance_test', 'mi_test'):
        out = out.sort_values(['_symbol_rank', '_overall'], ascending=[True, False], kind='stable')
    else:
        out = out.sort_values(['_symbol_rank', 'Features'], ascending=[True, True], kind='stable')

    out = out.drop(columns=['_overall', '_symbol_rank']).set_index(['Symbol', 'Features'])
    out.columns = pd.MultiIndex.from_tuples([tuple(c) for c in metric_cols])
    return out


feature_importance_tables = {}

for layer in LAYER_CONFIG:
    table = feature_importance_table(feature_importance_avg, layer, symbols=SYMBOLS, horizons=HORIZONS)
    feature_importance_tables[layer] = table

    label = LAYER_CONFIG[layer]['label']
    print(f"--- {layer} ({label}) — Feature Importance (MI / SHAP) ---")
    if table.empty:
        print('  No feature-importance rows found for this layer/symbol combination yet.\n')
    else:
        display(table)
        print()


### 12.5b Domain-level importance, generalized across symbols (RQ4 / RQ5)

The feature-level table above is useful for identifying individual top features, but the
paper's RQ4/RQ5 ask a *domain*-level question: does the relative importance of an information
**family** (market/technical, momentum, volatility/volume, cross-market, macroeconomic,
geopolitical, sentiment, sustainability, event, interaction, ...) change across horizons and
market segments?

To answer this with a single generalized number per (layer, horizon), the same averaging
principle as Section 9b is applied to feature importance:

1. Within each `(symbol, horizon)`, `shap_importance_test` is turned into a **share of total
   SHAP importance** (so magnitudes are comparable across symbols/layers regardless of scale).
2. Shares are summed by `domain` within each `(symbol, horizon)`.
3. The domain share is then **averaged across all three symbols**, requiring every symbol to be
   present (same `min_symbols` guard as `aggregate_across_symbols`).

The result is one row per `domain`, one column per horizon — directly showing whether, e.g.,
sentiment or sustainability information becomes more or less important at longer horizons,
generalized across NIFTY50 / BANKNIFTY / NIFTYNEXT50 rather than read off a single index.


In [ ]:
def domain_importance_table(df, layer, symbols=SYMBOLS, horizons=HORIZONS, min_symbols=None):
    """Average SHAP-importance *share* by information domain, per horizon,
    averaged across symbols -- the feature-importance analogue of
    `best_model_by_layer_horizon` above.
    """
    if min_symbols is None:
        min_symbols = len(symbols)

    x = df[df['layer'].eq(layer)].copy()
    x = x[x['symbol'].isin(symbols) & x['horizon'].isin(horizons)]
    if x.empty:
        return pd.DataFrame()

    # Share of SHAP importance within each (symbol, horizon), so magnitudes
    # are comparable across symbols/layers regardless of raw SHAP scale.
    totals = x.groupby(['symbol', 'horizon'])['shap_importance_test'].transform('sum')
    x['shap_share'] = np.where(totals > 0, x['shap_importance_test'] / totals, np.nan)

    domain_sym_h = (
        x.groupby(['symbol', 'horizon', 'domain'], as_index=False)['shap_share']
         .sum()
    )

    out = (
        domain_sym_h.groupby(['horizon', 'domain'], as_index=False)
        .agg(mean_share=('shap_share', 'mean'),
             std_share=('shap_share', 'std'),
             n_symbols=('symbol', 'nunique'))
    )
    out = out[out['n_symbols'] >= min_symbols]
    if out.empty:
        return pd.DataFrame()

    wide = out.pivot(index='domain', columns='horizon', values='mean_share')
    wide = wide.reindex(columns=[h for h in horizons if h in wide.columns])
    wide.columns = [f'Horizon = {h}' for h in wide.columns]

    if not wide.empty:
        wide = wide.loc[wide.mean(axis=1, skipna=True).sort_values(ascending=False).index]

    return wide


domain_importance_tables = {}

for layer in LAYER_CONFIG:
    table = domain_importance_table(feature_importance_avg, layer, symbols=SYMBOLS, horizons=HORIZONS)
    domain_importance_tables[layer] = table

    label = LAYER_CONFIG[layer]['label']
    print(f"--- {layer} ({label}) — domain share of SHAP importance by horizon (avg across {len(SYMBOLS)} symbols) ---")
    if table.empty:
        print('  No feature-importance rows found for all symbols on this layer yet.\n')
    else:
        display(table.round(4))
        print()


### 12.5 Convenience accessor

`feature_importance_tables` is a dict keyed by layer name, matching `publication_tables` from the leaderboard section. For example, the layer from your example path (`L1_1_DIR`):

In [ ]:
L1_1_DIR_feature_importance = feature_importance_tables.get('L1_1_DIR', pd.DataFrame())

print('L1_1_DIR — Feature Importance (Symbol x Feature x Horizon x [MI, SHAP])')
display(L1_1_DIR_feature_importance)


### 12.6 Save aggregated feature-importance outputs

Saved to a separate `Artifacts/feature_importance/feature_importance_aggregated/` folder,
mirroring how the leaderboard aggregation is kept separate from the per-run source files.
Both the Symbol × Feature tables (12.4) and the generalized Domain × Horizon tables (12.5b)
are saved.


In [ ]:
FEATURE_AGG_ROOT = FEATURE_IMPORTANCE_ROOT / 'feature_importance_aggregated'
FEATURE_AGG_ROOT.mkdir(parents=True, exist_ok=True)

# Raw consolidated long tables (per target/feature, and de-duplicated).
feature_importance_long.to_csv(FEATURE_AGG_ROOT / f'{symbols_tag}_mi_shap_test_all_raw.csv', index=False)
feature_importance_avg.to_csv(FEATURE_AGG_ROOT / f'{symbols_tag}_mi_shap_test_resolved_long.csv', index=False)

# Publication-style Symbol x Feature table for every layer.
for layer, table in feature_importance_tables.items():
    if not table.empty:
        table.to_csv(FEATURE_AGG_ROOT / f'{symbols_tag}_{layer}_feature_importance_publication.csv')

# Generalized Domain x Horizon tables (averaged across symbols), Section 12.5b.
for layer, table in domain_importance_tables.items():
    if not table.empty:
        table.to_csv(FEATURE_AGG_ROOT / f'{symbols_tag}_{layer}_domain_importance_by_horizon.csv')

print('Saved aggregated feature-importance outputs to:')
print(FEATURE_AGG_ROOT)


### 12.6b Top features per domain, for every output variable (new)

With ~100 raw features, a flat "top 15" list is usually dominated by 1-2 information
domains and hides the others. This instead takes the **top 5 features within each
domain** (averaged across all symbols and horizons), for **every** output variable at
once — a compact overview you can put in the paper without needing one figure per
target/horizon like Section 13.2 gives you for a single slice.

In [ ]:
# ------------------------------------------------------------------
# 12.6b Top-N features PER DOMAIN, for every output variable (layer).
# ------------------------------------------------------------------

FEATURE_TOP_N_PER_DOMAIN = 5  # change to 8-10 if your paper has room for more.


def top_features_per_domain(df, layer, top_n=FEATURE_TOP_N_PER_DOMAIN,
                             symbols=SYMBOLS, horizons=HORIZONS, metric='shap_importance_test'):
    """Top `top_n` features WITHIN EACH DOMAIN for one layer, averaged across
    all symbols and horizons requested -- guarantees every domain gets fair
    representation instead of a flat top-N being dominated by 1-2 domains.
    """
    x = df[(df['layer'] == layer) & df['symbol'].isin(symbols) & df['horizon'].isin(horizons)]
    if x.empty:
        return pd.DataFrame()
    grouped = x.groupby(['feature', 'domain'], as_index=False)[metric].mean().rename(columns={metric: 'mean_value'})
    grouped['rank_in_domain'] = grouped.groupby('domain')['mean_value'].rank(ascending=False, method='first')
    return grouped[grouped['rank_in_domain'] <= top_n].sort_values(['domain', 'mean_value'], ascending=[True, True])


n_layers = len(LAYER_CONFIG)
n_cols = 2
n_rows = int(np.ceil(n_layers / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, 4.3 * n_rows), squeeze=False)

for idx, (layer, cfg) in enumerate(LAYER_CONFIG.items()):
    ax = axes[idx // n_cols][idx % n_cols]
    feats = top_features_per_domain(feature_importance_avg, layer)
    if feats.empty:
        ax.axis('off')
        continue

    y = np.arange(len(feats))
    colors = feats['domain'].map(DOMAIN_COLORS)
    ax.barh(y, feats['mean_value'], color=colors, edgecolor='black', linewidth=0.3)
    ax.set_yticks(y)
    ax.set_yticklabels(feats['feature'], fontsize=6.5)
    ax.set_title(TARGET_LABELS.get(layer, layer), fontsize=10)
    ax.tick_params(axis='x', labelsize=7)
    ax.grid(axis='x', alpha=0.3)

for idx in range(n_layers, n_rows * n_cols):
    axes[idx // n_cols][idx % n_cols].axis('off')

fig.legend(
    handles=domain_legend_handles(), title='Domain', loc='lower center',
    ncol=min(len(DOMAIN_ORDER), 6), bbox_to_anchor=(0.5, -0.02), fontsize=8,
)
fig.suptitle(f'Top {FEATURE_TOP_N_PER_DOMAIN} Features per Domain -- All Output Variables', fontsize=14, y=1.01)

fig.tight_layout(rect=[0, 0.05, 1, 1])
save_fig(fig, FEATURE_AGG_ROOT, 'top_features_per_domain_all_targets')
plt.show()

print(
    f"Suggested caption: \"Top {FEATURE_TOP_N_PER_DOMAIN} features within each information "
    "domain, by mean SHAP importance averaged across symbols and horizons, for every "
    "forecasting target.\"\n"
)


### 12.6c Domain contribution share, for every output variable (new)

The feature-level view above answers "which features matter"; this answers the
complementary RQ4/RQ5 question at the **domain** level — "which information family
(momentum, volatility, macro, ...) drives each output variable overall?" One 100%-stacked
bar per output variable, averaged across horizons, built directly from
`domain_importance_tables` (Section 12.5b) so the numbers are identical to that table —
this is purely a visualization of it.

In [ ]:
# ------------------------------------------------------------------
# 12.6c Domain contribution (SHAP share), averaged across horizons,
# for every output variable -- one stacked bar per layer.
# ------------------------------------------------------------------

domain_layer_share = {}
for layer, table in domain_importance_tables.items():
    if table.empty:
        continue
    domain_layer_share[layer] = table.mean(axis=1, skipna=True)  # avg share across horizons

if not domain_layer_share:
    print("No domain-importance tables available yet -- skipping the contribution plot.")
else:
    domain_layer_df = pd.DataFrame(domain_layer_share).reindex(index=DOMAIN_ORDER)
    domain_layer_df.columns = [TARGET_LABELS.get(l, l) for l in domain_layer_df.columns]
    # Re-normalize each column to sum to 1 (guards against a little missing mass
    # from partial symbol/horizon coverage in Section 12.5b).
    col_sums = domain_layer_df.sum(axis=0, skipna=True)
    domain_layer_df_norm = domain_layer_df.div(col_sums, axis=1)

    targets = domain_layer_df_norm.columns.tolist()
    x = np.arange(len(targets))
    bottom = np.zeros(len(targets))

    fig, ax = plt.subplots(figsize=(11, 6.5))
    for domain in domain_layer_df_norm.index:
        vals = domain_layer_df_norm.loc[domain].fillna(0).values
        ax.bar(x, vals, bottom=bottom, color=DOMAIN_COLORS.get(domain, '0.7'),
               edgecolor='black', linewidth=0.3, label=domain)
        for xi, (v, b) in enumerate(zip(vals, bottom)):
            if v > 0.06:  # only label segments big enough to read
                ax.text(xi, b + v / 2, f'{v * 100:.0f}%', ha='center', va='center', fontsize=7, color='white')
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(targets, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Share of Mean SHAP Importance', fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_title('Feature-Domain Contribution by Output Variable (averaged across horizons)', fontsize=13)
    ax.legend(title='Domain', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

    fig.tight_layout()
    save_fig(fig, FEATURE_AGG_ROOT, 'domain_contribution_by_target')
    plt.show()

    print(
        "Suggested caption: \"Share of mean SHAP importance contributed by each information "
        "domain, per output variable, averaged across symbols and horizons.\"\n"
    )


### 12.7 All features, category-wise, x all output variables (SHAP) -- single table

One consolidated table: every feature that appears in `feature_importance_avg`
(grouped/sorted by information **domain**, i.e. category), with one **column
per output variable** (every layer in `LAYER_CONFIG`) holding the mean
**SHAP importance**. This is the feature-level analogue of `publication_tables`
(Section 6) and `domain_importance_tables` (Section 12.5b) combined into one
view: rows = feature (by domain), columns = target/output variable.

Because this collapses across both **symbol** and **horizon**, a cell is only
filled in if the feature has `shap_importance_test` for every symbol x horizon
combination requested (`min_coverage`, defaults to `len(symbols) * len(horizons)`)
-- partial coverage is left blank (`NaN`) rather than silently averaged over
fewer observations, matching the rest of the notebook's convention.


**Note (this revision):** because this table has one row per *feature* (up to ~100 rows),
it is kept as a reference table / CSV export only -- the full heatmap of it has been
removed from the notebook. For a paper-ready figure, use **12.6b** (top features per
domain, all output variables) and **12.6c** (domain contribution share, all output
variables) instead -- both summarize this same underlying data at a scale that actually
fits on a page.

In [ ]:
def build_feature_by_target_table(df, symbols=SYMBOLS, horizons=HORIZONS, layers=None,
                                   metric='shap_importance_test', min_coverage=None):
    """Build a Domain/Feature x Output-Variable table of `metric`.

    Averages `metric` for each (layer, feature) across every symbol and
    horizon requested, so the result is a single value per feature per
    output variable (layer) -- not split by symbol or horizon. Rows are
    grouped by `domain` (feature category) and, within each domain, sorted
    by overall mean importance (descending).

    min_coverage: minimum number of (symbol, horizon) observations required
    for a (layer, feature) cell to be reported. Defaults to requiring full
    coverage across every symbol and horizon supplied.
    """
    if layers is None:
        layers = list(LAYER_CONFIG.keys())
    if min_coverage is None:
        min_coverage = len(symbols) * len(horizons)

    x = df[
        df['symbol'].isin(symbols)
        & df['horizon'].isin(horizons)
        & df['layer'].isin(layers)
    ].copy()
    if x.empty:
        return pd.DataFrame()

    grouped = x.groupby(['layer', 'feature'], as_index=False).agg(
        mean_value=(metric, 'mean'),
        n_obs=(metric, 'size'),
        domain=('domain', 'first'),
    )
    grouped = grouped[grouped['n_obs'] >= min_coverage]
    if grouped.empty:
        return pd.DataFrame()

    wide = grouped.pivot(index='feature', columns='layer', values='mean_value')

    # Column order/labels follow LAYER_CONFIG, renamed to the human-readable
    # target labels used elsewhere in the notebook.
    ordered_layers = [l for l in LAYER_CONFIG if l in wide.columns]
    wide = wide[ordered_layers]
    wide.columns = [TARGET_LABELS.get(l, l) for l in ordered_layers]

    # Attach domain (feature category) as the outer row index.
    domain_lookup = grouped.drop_duplicates('feature').set_index('feature')['domain']
    wide.insert(0, 'Domain', wide.index.map(domain_lookup))
    wide = wide.reset_index().rename(columns={'feature': 'Feature'})

    # Sort by domain, then by each feature's overall mean importance
    # (averaged across all output variables) within its domain, descending.
    value_cols = [c for c in wide.columns if c not in ('Domain', 'Feature')]
    wide['_overall'] = wide[value_cols].mean(axis=1, skipna=True)
    wide = wide.sort_values(['Domain', '_overall'], ascending=[True, False]).drop(columns='_overall')

    return wide.set_index(['Domain', 'Feature'])


feature_by_target_shap = build_feature_by_target_table(
    feature_importance_avg, symbols=SYMBOLS, horizons=HORIZONS, metric='shap_importance_test'
)

print('All features (by domain) x all output variables -- mean SHAP importance:')
if feature_by_target_shap.empty:
    print('  No (layer, feature) combination has full symbol x horizon coverage yet.')
else:
    display(feature_by_target_shap.round(4))


In [ ]:
# Same table for Mutual Information (MI), in case both metrics are needed
# side by side.

feature_by_target_mi = build_feature_by_target_table(
    feature_importance_avg, symbols=SYMBOLS, horizons=HORIZONS, metric='mi_test'
)

print('All features (by domain) x all output variables -- mean MI:')
if feature_by_target_mi.empty:
    print('  No (layer, feature) combination has full symbol x horizon coverage yet.')
else:
    display(feature_by_target_mi.round(4))


In [ ]:
feature_by_target_shap.to_csv(
    FEATURE_AGG_ROOT / f'{symbols_tag}_feature_by_target_shap.csv'
)
feature_by_target_mi.to_csv(
    FEATURE_AGG_ROOT / f'{symbols_tag}_feature_by_target_mi.csv'
)

print('Saved feature x output-variable tables to:')
print(FEATURE_AGG_ROOT / f'{symbols_tag}_feature_by_target_shap.csv')
print(FEATURE_AGG_ROOT / f'{symbols_tag}_feature_by_target_mi.csv')


### 12.8 Descriptive statistics (`df.describe()`) for the top-10 features per output variable, averaged over symbols

For **every layer (output variable)** in `LAYER_CONFIG`:

1. Features are ranked by **mean SHAP importance**, averaged across every symbol and horizon
   requested (same ranking rule used in Sections 12.6b / 13.1).
2. The **top 10** features are kept.
3. `df.describe()` is run on `mi_test` and `shap_importance_test` for those 10 features,
   computed **across the (symbol, horizon) rows** -- so instead of a single averaged number
   you get `count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`, showing how stable each
   feature's importance is across the different symbols/horizons rather than just its point estimate.


In [ ]:
# ------------------------------------------------------------------
# 12.8 Descriptive statistics (df.describe()) for the top-10 features
# per output variable, averaged over symbols.
# ------------------------------------------------------------------

FEATURE_DESCRIBE_TOP_N = 10


def top_n_features_for_layer(df, layer, top_n=FEATURE_DESCRIBE_TOP_N,
                              symbols=SYMBOLS, horizons=HORIZONS,
                              metric='shap_importance_test'):
    """Top `top_n` features for one layer, ranked by mean `metric`
    averaged across all symbols and horizons requested."""
    x = df[(df['layer'] == layer) & df['symbol'].isin(symbols) & df['horizon'].isin(horizons)]
    if x.empty:
        return []
    ranked = (
        x.groupby('feature')[metric]
        .mean()
        .sort_values(ascending=False)
    )
    return ranked.head(top_n).index.tolist()


def describe_top_features(df, layer, top_n=FEATURE_DESCRIBE_TOP_N,
                           symbols=SYMBOLS, horizons=HORIZONS,
                           metrics=('mi_test', 'shap_importance_test')):
    """df.describe() for the top-N features of one layer, computed across
    all (symbol, horizon) rows -- i.e. the distribution of each feature's
    importance over symbols, not just its mean."""
    top_feats = top_n_features_for_layer(df, layer, top_n, symbols, horizons)
    if not top_feats:
        return pd.DataFrame()

    x = df[
        (df['layer'] == layer)
        & df['feature'].isin(top_feats)
        & df['symbol'].isin(symbols)
        & df['horizon'].isin(horizons)
    ]

    # One describe() block per feature (rows), stacked; metrics as the
    # outer column level. Reindexed so rows stay in top-N rank order
    # (groupby would otherwise sort features alphabetically).
    out = (
        x.groupby('feature')[list(metrics)]
        .describe()
        .reindex(top_feats)
    )
    return out


feature_describe_tables = {}

for layer in LAYER_CONFIG:
    table = describe_top_features(feature_importance_avg, layer)
    feature_describe_tables[layer] = table

    label = LAYER_CONFIG[layer]['label']
    print(f"--- {layer} ({label}) \u2014 describe() for top {FEATURE_DESCRIBE_TOP_N} features (across symbols) ---")
    if table.empty:
        print('  No feature-importance rows found for this layer yet.\n')
    else:
        display(table.round(4))
        print()


In [ ]:
# ------------------------------------------------------------------
# 12.8b Save the per-layer top-feature describe() tables
# ------------------------------------------------------------------

for layer, table in feature_describe_tables.items():
    if not table.empty:
        table.to_csv(
            FEATURE_AGG_ROOT / f'{symbols_tag}_{layer}_top{FEATURE_DESCRIBE_TOP_N}_features_describe.csv'
        )

print(f'Saved per-layer top-{FEATURE_DESCRIBE_TOP_N} feature describe() tables to:')
print(FEATURE_AGG_ROOT)


## 12.9 TRAIN-data descriptive statistics (`df.describe()`) for the top-10 **selected** features — Symbol × Layer × Horizon

Section 12.8 above runs `df.describe()` on the *importance scores* (`mi_test`,
`shap_importance_test`) of the top features. This section instead goes back to the source:
it loads the **same TRAIN split parquet files** used by the `Phase05_Unified_L1/L2/L3`
notebooks (`{symbol}_features_train.parquet` under `Data/features_v6/`), reads the **exact
train-only selected feature list** each of those notebooks wrote out per
`(symbol, layer, horizon)` (`Data/feature_selection_v6/{symbol}/{layer}/{symbol}_selection_{layer}_{h}d.txt`),
keeps the top 10, and runs `df.describe()` directly on those features' **real values** in the
TRAIN data.

This gives the actual distribution (`count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`,
`max`) of each selected feature as it appears in the data used to fit the models — not just
how important it was scored to be. Nothing is re-selected or re-fit here; this only reads
artifacts the Unified notebooks already produced, exactly like the rest of this notebook.

In [ ]:
# ------------------------------------------------------------------
# 12.9a Paths -- same Phase-03 feature matrices + Phase-05 feature-selection
# outputs read by the Unified L1/L2/L3 notebooks (their Section 2 / 3.1).
# Nothing is retrained or re-selected here; only existing artifacts are read.
# ------------------------------------------------------------------
DATA_ROOT = Path(GDRIVE_BASE) / 'Data'
FEAT_DIR  = DATA_ROOT / 'features_v6'           # Phase-03 {symbol}_features_{split}.parquet
FS_DIR    = DATA_ROOT / 'feature_selection_v6'  # Phase-05 {symbol}_selection_{layer}_{h}d.txt

TRAIN_DESCRIBE_TOP_N = 10

print('FEAT_DIR:', FEAT_DIR)
print('FS_DIR  :', FS_DIR)


In [ ]:
# ------------------------------------------------------------------
# 12.9b Load TRAIN data (cached per symbol) -- identical source/pattern to
# load_raw_splits() in the Unified notebooks, restricted to the 'train' split.
# ------------------------------------------------------------------
_TRAIN_DATA_CACHE = {}


def load_train_data(symbol):
    """Loads {symbol}_features_train.parquet -- the exact TRAIN split the Unified
    L1/L2/L3 notebooks use for train-only feature selection and model fitting.
    Cached per symbol so repeated (layer, horizon) lookups don't re-read the file."""
    if symbol in _TRAIN_DATA_CACHE:
        return _TRAIN_DATA_CACHE[symbol]

    path = FEAT_DIR / symbol / f'{symbol}_features_train.parquet'
    if not path.exists():
        print(f'  ⚠️  Train parquet not found: {path}')
        _TRAIN_DATA_CACHE[symbol] = None
        return None

    df = pd.read_parquet(path)
    print(f'  Loaded {symbol}/train  shape={df.shape}')
    _TRAIN_DATA_CACHE[symbol] = df
    return df


print('Pre-loading TRAIN splits for all symbols...')
for _sym in SYMBOLS:
    load_train_data(_sym)


In [ ]:
# ------------------------------------------------------------------
# 12.9c Selected features per (symbol, layer, horizon) -- read straight from the
# .txt file the Unified notebook wrote via save_selected_features(). That list is
# already sorted best-first by FINAL_SELECTION_BASIS (MI / SHAP / both), so the
# first N lines ARE the top-N selected features. Falls back to the aggregated
# mi_shap_test.csv ranking (feature_importance_avg, Section 12.3) if the .txt
# file isn't available, so this section still runs off whatever exists.
# ------------------------------------------------------------------
def load_selected_features(symbol, layer, horizon, top_n=TRAIN_DESCRIBE_TOP_N):
    txt_path = FS_DIR / symbol / layer / f'{symbol}_selection_{layer}_{horizon}d.txt'
    if txt_path.exists():
        feats = [ln.strip() for ln in txt_path.read_text().splitlines() if ln.strip()]
        return feats[:top_n], str(txt_path)

    if not feature_importance_avg.empty:
        x = feature_importance_avg[
            feature_importance_avg['symbol'].eq(symbol)
            & feature_importance_avg['layer'].eq(layer)
            & feature_importance_avg['horizon'].eq(horizon)
        ]
        if not x.empty:
            feats = (
                x.sort_values('shap_importance_test', ascending=False)['feature']
                .head(top_n).tolist()
            )
            return feats, 'feature_importance_avg (fallback -- .txt selection file missing)'

    return [], None


In [ ]:
# ------------------------------------------------------------------
# 12.9d df.describe() on the ACTUAL TRAIN values (not importance scores) for the
# top-N selected features of one (symbol, layer, horizon).
# ------------------------------------------------------------------
def describe_train_features(symbol, layer, horizon, top_n=TRAIN_DESCRIBE_TOP_N):
    """Real TRAIN-split df.describe() for the top-`top_n` selected features of one
    (symbol, layer, horizon). Inf is treated as missing so it doesn't distort
    min/max; values are otherwise exactly as they appear in the Phase-03 parquet
    (no imputation), so `count` also doubles as a missingness diagnostic."""
    feats, source = load_selected_features(symbol, layer, horizon, top_n)
    if not feats:
        return pd.DataFrame(), []

    df_train = load_train_data(symbol)
    if df_train is None:
        return pd.DataFrame(), feats

    present = [f for f in feats if f in df_train.columns]
    missing_cols = [f for f in feats if f not in df_train.columns]
    if missing_cols:
        print(f'  ⚠️  {symbol}/{layer}/h{horizon}d: {len(missing_cols)} selected '
              f'feature(s) not found in train parquet: {missing_cols}')
    if not present:
        return pd.DataFrame(), feats

    x = df_train[present].replace([np.inf, -np.inf], np.nan)
    desc = x.describe().T
    desc.insert(0, 'selection_rank', [feats.index(f) + 1 for f in present])
    desc.insert(0, 'horizon', horizon)
    desc.insert(0, 'layer', layer)
    desc.insert(0, 'symbol', symbol)
    desc.index.name = 'feature'
    return desc.reset_index(), feats


In [ ]:
# ------------------------------------------------------------------
# 12.9e Run for every (symbol, layer, horizon) combination
# ------------------------------------------------------------------
train_describe_frames = []
train_describe_tables = {}   # (symbol, layer, horizon) -> describe() DataFrame

for layer in LAYER_CONFIG:
    for symbol in SYMBOLS:
        for horizon in HORIZONS:
            desc, feats = describe_train_features(symbol, layer, horizon, TRAIN_DESCRIBE_TOP_N)
            train_describe_tables[(symbol, layer, horizon)] = desc
            if not desc.empty:
                train_describe_frames.append(desc)

train_describe_long = (
    pd.concat(train_describe_frames, ignore_index=True)
    if train_describe_frames else pd.DataFrame()
)

n_combos = len(LAYER_CONFIG) * len(SYMBOLS) * len(HORIZONS)
print(f'Built TRAIN-data df.describe() for {len(train_describe_frames)} / {n_combos} '
      f'(symbol, layer, horizon) combinations.')
display(train_describe_long.head(20))


In [ ]:
# ------------------------------------------------------------------
# 12.9f Example view -- one (symbol, layer, horizon), top-10 selected features
# ------------------------------------------------------------------
_example_layer   = next(iter(LAYER_CONFIG))
_example_symbol  = SYMBOLS[0]
_example_horizon = HORIZONS[0]

_example_key = (_example_symbol, _example_layer, _example_horizon)
_example_table = train_describe_tables.get(_example_key, pd.DataFrame())

print(f'--- Example: {_example_symbol} / {_example_layer} / h{_example_horizon}d --- '
      f'df.describe() on TRAIN values for the top {TRAIN_DESCRIBE_TOP_N} selected features ---')
if _example_table.empty:
    print('  No data available yet for this combination.')
else:
    display(
        _example_table
        .set_index('feature')[['selection_rank', 'count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
        .round(4)
    )


In [ ]:
# ------------------------------------------------------------------
# 12.9g Save the TRAIN-data top-N feature describe() tables
# ------------------------------------------------------------------
TRAIN_DESCRIBE_ROOT = FEATURE_AGG_ROOT / 'train_feature_describe'
TRAIN_DESCRIBE_ROOT.mkdir(parents=True, exist_ok=True)

if not train_describe_long.empty:
    train_describe_long.to_csv(
        TRAIN_DESCRIBE_ROOT / f'{symbols_tag}_top{TRAIN_DESCRIBE_TOP_N}_train_features_describe_all.csv',
        index=False,
    )
    for layer, g in train_describe_long.groupby('layer'):
        g.to_csv(
            TRAIN_DESCRIBE_ROOT / f'{symbols_tag}_{layer}_top{TRAIN_DESCRIBE_TOP_N}_train_features_describe.csv',
            index=False,
        )

print(f'Saved TRAIN-data df.describe() tables (top-{TRAIN_DESCRIBE_TOP_N} selected features '
      f'per symbol/layer/horizon) to:')
print(TRAIN_DESCRIBE_ROOT)


### Notes (Section 12.9)

- This reuses the **exact TRAIN parquet files and selected-feature `.txt` lists** the
  `Phase05_Unified_L1/L2/L3` notebooks already produced — no feature selection, imputation,
  or model fitting happens here.
- `count` in the describe table reflects real missingness in the raw TRAIN column (only `inf`
  is converted to `NaN`; other `NaN`s are left as-is), unlike the model-fitting pipeline which
  median-imputes before training.
- If a `.txt` selection file is missing for a given `(symbol, layer, horizon)`, the top-10 is
  derived instead from the aggregated `mi_shap_test.csv` SHAP ranking already loaded in
  Section 12 (`feature_importance_avg`) — check the printed `source` / any `⚠️` warnings
  above if a table looks unexpectedly empty.

### Notes (Feature Importance section)

- Only **`mi_shap_test.csv`** is read (TEST split); `mi_shap_train.csv` is intentionally ignored,
  matching the leaderboard's TEST-only convention.
- The source file has one row per **(target, feature)** — not per model — so `mi_test` and
  `shap_importance_test` are used as-is (de-duplicated defensively in Section 12.3, not
  averaged across models).
- Missing files/horizons/symbols/layers show up as missing (`NaN`) in `feature_file_inventory` /
  the tables; they are never silently filled with zero.
- `feature_importance_tables` (12.4) contains one Symbol × Feature × (Horizon, Metric)
  DataFrame **per layer**, exactly like `publication_tables` for the leaderboards.
- `domain_importance_tables` (12.5b) contains one Domain × Horizon DataFrame **per layer**,
  generalized (mean-averaged) across all three symbols — the feature-importance analogue of
  `best_model_summary` from Section 9b, and the most direct way to answer RQ4/RQ5 (how the
  contribution of market / macro / geopolitical / sentiment / sustainability information
  changes across horizons and market segments).


## 13. Best-Model Feature Explainability

Links the generalized "best model" selection (Section 9b / 11) to the
feature-importance tables (Section 12), so you can see *what was driving*
the target that each winning model was chosen on.

Important caveat: `mi_test` / `shap_importance_test` are computed once per
`(target, feature)` from a train-only surrogate model (see Section 3.4.4 of
the report) -- they are **not** SHAP values of the winning model itself, and
they don't vary by model. What this section shows is "the feature landscape
for the target the best model was chosen on", not "the winning model's own
SHAP values".

- **13.1 Summarized** -- one row per (layer, horizon): best model, top
  information domain, and top-3 features, all in a single table.
- **13.2 Detailed** -- a horizontal bar chart of the top-N features (by
  mean SHAP importance, averaged across symbols) for one chosen
  (layer, horizon), colored by information domain.
- **13.3 Detailed grid** -- the same chart as 13.2, but as small multiples
  across every horizon for one chosen layer, to see which features persist
  vs. fade as the horizon lengthens.


In [ ]:
# ============================================================
# 13.1 Summarized: best model + top domain + top-3 features
# ============================================================

def top_features_for_layer_horizon(df, layer, horizon, symbols=SYMBOLS, top_n=10,
                                    metric='shap_importance_test', min_symbols=None):
    """Average `metric` across symbols for one (layer, horizon), ranked descending.

    Only features observed for at least `min_symbols` distinct symbols are
    kept (defaults to requiring every symbol), mirroring
    `aggregate_across_symbols` from Section 9b.
    """
    if min_symbols is None:
        min_symbols = len(symbols)

    x = df[
        (df['layer'] == layer)
        & (df['horizon'] == horizon)
        & df['symbol'].isin(symbols)
    ].copy()
    if x.empty:
        return pd.DataFrame()

    grouped = x.groupby('feature', as_index=False).agg(
        mean_value=(metric, 'mean'),
        std_value=(metric, 'std'),
        n_symbols=('symbol', 'nunique'),
        domain=('domain', 'first'),
    )
    grouped = grouped[grouped['n_symbols'] >= min_symbols]
    return grouped.sort_values('mean_value', ascending=False).head(top_n).reset_index(drop=True)


def top_domain_for_layer_horizon(layer, horizon):
    """Look up the top-share domain for one (layer, horizon) from
    `domain_importance_tables` (Section 12.5b)."""
    table = domain_importance_tables.get(layer, pd.DataFrame())
    col = f'Horizon = {int(horizon)}'
    if table.empty or col not in table.columns:
        return None, np.nan
    s = table[col].dropna()
    if s.empty:
        return None, np.nan
    return s.idxmax(), s.max()


summary_rows = []
for _, row in best_model_summary.iterrows():
    layer, horizon, model = row['layer'], row['horizon'], row['model']

    top_feats = top_features_for_layer_horizon(
        feature_importance_avg, layer, horizon, top_n=3
    )
    top_domain, top_domain_share = top_domain_for_layer_horizon(layer, horizon)

    summary_rows.append({
        'Layer': layer,
        'Target': TARGET_LABELS.get(layer, row.get('label', layer)),
        'Horizon': int(horizon),
        'Best Model': model,
        'Top Domain': top_domain,
        'Top Domain Share': (
            round(top_domain_share, 3) if pd.notna(top_domain_share) else np.nan
        ),
        'Top 3 Features (SHAP)': (
            ', '.join(top_feats['feature']) if not top_feats.empty else None
        ),
    })

best_model_feature_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(['Layer', 'Horizon'])
    .reset_index(drop=True)
)

print('Best model + top feature summary (one row per layer x horizon):')
display(best_model_feature_summary)


def top_features_per_domain_for_layer_horizon(df, layer, horizon, top_n=5,
                                               symbols=SYMBOLS, metric='shap_importance_test',
                                               min_symbols=None):
    """Like `top_features_per_domain` (Section 12.6b) but scoped to a single
    (layer, horizon) instead of averaged across all horizons -- used by the
    detail plots (13.2 / 13.3) so every domain is represented instead of a
    flat top-N being dominated by 1-2 domains.
    """
    if min_symbols is None:
        min_symbols = len(symbols)

    x = df[(df['layer'] == layer) & (df['horizon'] == horizon) & df['symbol'].isin(symbols)].copy()
    if x.empty:
        return pd.DataFrame()

    grouped = x.groupby(['feature', 'domain'], as_index=False).agg(
        mean_value=(metric, 'mean'),
        std_value=(metric, 'std'),
        n_symbols=('symbol', 'nunique'),
    )
    grouped = grouped[grouped['n_symbols'] >= min_symbols]
    grouped['rank_in_domain'] = grouped.groupby('domain')['mean_value'].rank(ascending=False, method='first')
    return grouped[grouped['rank_in_domain'] <= top_n].sort_values(['domain', 'mean_value'], ascending=[True, False])


### 13.2 Feature importance for the winning model's target (updated: top 5 per domain)

Shows the top 5 features **within each domain** (not a flat top-15) for one
target/horizon, titled with the model that actually won that slot — so every domain gets
fair representation instead of 1-2 dominant domains crowding out the rest. Bars use the
**global** domain color map from Section 12.1b, so "Momentum" (for example) is always the
same color here as in every other feature plot in the notebook.

In [ ]:
# ============================================================
# 13.2 Detailed: feature-importance bar chart for one (layer, horizon)
#
# Change `detail_layer` / `detail_horizon` below to inspect a different
# target/horizon. Defaults to the first layer in LAYER_CONFIG at the
# shortest horizon. Shows the top `detail_top_n_per_domain` features
# WITHIN EACH DOMAIN (see 13.1's `top_features_per_domain_for_layer_horizon`),
# colored using the GLOBAL domain color map (Section 12.1b).
# ============================================================

detail_layer = next(iter(LAYER_CONFIG))
detail_horizon = HORIZONS[0]
detail_top_n_per_domain = 5

detail_features = top_features_per_domain_for_layer_horizon(
    feature_importance_avg, detail_layer, detail_horizon, top_n=detail_top_n_per_domain
)

if detail_features.empty:
    print(f'No feature-importance rows for {detail_layer} / h={detail_horizon}d yet.')
else:
    best_row = best_model_summary[
        (best_model_summary['layer'] == detail_layer)
        & (best_model_summary['horizon'] == detail_horizon)
    ]
    best_model_name = best_row['model'].iloc[0] if not best_row.empty else 'n/a'

    plot_data = detail_features.reset_index(drop=True)  # already domain-grouped, ascending within domain
    y = np.arange(len(plot_data))
    colors = plot_data['domain'].map(DOMAIN_COLORS)

    fig, ax = plt.subplots(figsize=(9, max(4.5, len(plot_data) * 0.32)))

    ax.barh(
        y, plot_data['mean_value'], xerr=plot_data['std_value'].fillna(0),
        color=colors, edgecolor='black', linewidth=0.4,
    )

    ax.set_yticks(y)
    ax.set_yticklabels(plot_data['feature'], fontsize=8.5)
    ax.set_xlabel('Mean SHAP Importance (avg. across symbols)', fontsize=10)
    ax.set_title(
        f"Top {detail_top_n_per_domain} Features per Domain -- "
        f"{TARGET_LABELS.get(detail_layer, detail_layer)}, h={detail_horizon}d\n"
        f"(winning model: {best_model_name})",
        fontsize=12,
    )
    ax.grid(axis='x', alpha=0.30)

    present_domains = sorted(plot_data['domain'].dropna().unique())
    ax.legend(handles=domain_legend_handles(present_domains), title='Domain', fontsize=8, loc='lower right')

    fig.tight_layout()
    filename = f"feature_detail_{detail_layer.lower()}_h{detail_horizon}d"
    save_fig(fig, FEATURE_AGG_ROOT, filename)
    plt.show()


### 13.3 Feature importance across all horizons for one target (updated: top 3 per domain)

Same idea as 13.2, but small-multiples across every horizon for one target — useful for
seeing whether the same handful of features stay dominant as the horizon lengthens, or
whether the driving domain shifts. Uses a smaller per-domain cap (3) than 13.2 since this
view already has up to 5 panels sharing the figure. Domain colors are the same global map
used throughout Sections 12-13.

In [ ]:
# ============================================================
# 13.3 Detailed grid: feature importance across all horizons, one layer
#
# Same chart as 13.2, but as small multiples across every horizon for a
# single target. Reuses `detail_layer` from 13.2; change `grid_layer`
# below to look at a different target instead. Uses top-N-per-domain
# (smaller N than 13.2, since there are up to 5 panels here) and the
# GLOBAL domain color map so colors match 13.2 and every other plot.
# ============================================================

grid_layer = detail_layer
grid_top_n_per_domain = 3

horizons_with_data = [
    h for h in HORIZONS
    if not top_features_per_domain_for_layer_horizon(feature_importance_avg, grid_layer, h, top_n=1).empty
]

if not horizons_with_data:
    print(f'No feature-importance rows found for {grid_layer} at any horizon yet.')
else:
    n_cols = min(3, len(horizons_with_data))
    n_rows = int(np.ceil(len(horizons_with_data) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.2 * n_rows), squeeze=False)

    layer_domains = sorted(
        feature_importance_avg.loc[feature_importance_avg['layer'] == grid_layer, 'domain'].dropna().unique()
    )

    for idx, h in enumerate(horizons_with_data):
        ax = axes[idx // n_cols][idx % n_cols]

        feats = top_features_per_domain_for_layer_horizon(
            feature_importance_avg, grid_layer, h, top_n=grid_top_n_per_domain
        ).reset_index(drop=True)
        y = np.arange(len(feats))
        colors = feats['domain'].map(DOMAIN_COLORS)

        ax.barh(y, feats['mean_value'], color=colors, edgecolor='black', linewidth=0.4)
        ax.set_yticks(y)
        ax.set_yticklabels(feats['feature'], fontsize=7)
        ax.set_title(f'h={h}d', fontsize=10)
        ax.grid(axis='x', alpha=0.30)

    for idx in range(len(horizons_with_data), n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].axis('off')

    fig.legend(
        handles=domain_legend_handles(layer_domains), title='Domain', loc='lower center',
        ncol=min(len(layer_domains), 5), fontsize=8, bbox_to_anchor=(0.5, -0.04),
    )

    fig.suptitle(
        f'Top {grid_top_n_per_domain} Features per Domain, by Horizon -- {TARGET_LABELS.get(grid_layer, grid_layer)}',
        fontsize=14,
    )

    fig.tight_layout(rect=[0, 0.05, 1, 1])
    save_fig(fig, FEATURE_AGG_ROOT, f"feature_detail_grid_{grid_layer.lower()}")
    plt.show()


In [ ]:
# ------------------------------------------------------------------
# 13.4 Save the best-model feature summary (13.1)
# ------------------------------------------------------------------

best_model_feature_summary.to_csv(
    FEATURE_AGG_ROOT / f'{symbols_tag}_best_model_feature_summary.csv', index=False
)

print('Saved best-model feature summary to:')
print(FEATURE_AGG_ROOT / f'{symbols_tag}_best_model_feature_summary.csv')


## 13.5 Combined best-model & top-feature dashboard (new)

Sections 11 and 13 answer two separate questions — "which model wins?" and "which feature
drives it?" — with separate plots. For a paper, it's often more useful to see **both
answers on one figure**: the winning model (bar height + color) *and* its single most
important feature (label on the bar), for every target and horizon at once.

**How to read it.** Bar height is the winning model's mean TEST score (Macro-F1 for
classification, RMSE for regression — read the two panels separately, since a taller bar
means "better" in the left panel but "worse" in the right one). Bar color is the model's
category (same legend as Section 11). The rotated label on top of each bar is the single
most important feature (by mean SHAP) for that (target, horizon) cell, taken from
`best_model_feature_summary` in Section 13.1.

This is the plot to reach for when the paper needs one figure that ties "best model" and
"best feature" together, rather than asking the reader to cross-reference two tables.

In [ ]:
# ============================================================
# 13.5 Combined Best-Model + Top-Feature dashboard
# ============================================================

dash_df = plot_df.merge(
    best_model_feature_summary.rename(columns={
        'Layer': 'layer', 'Horizon': 'horizon',
    })[['layer', 'horizon', 'Top 3 Features (SHAP)']],
    on=['layer', 'horizon'], how='left',
)
dash_df['top_feature'] = dash_df['Top 3 Features (SHAP)'].apply(
    lambda s: s.split(',')[0].strip() if isinstance(s, str) and s else None
)

if dash_df.empty:
    print("Nothing to plot -- plot_df / best_model_feature_summary is empty.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, task_name, ylabel in zip(
        axes, ["classification", "regression"], ["Mean TEST Macro-F1", "Mean TEST RMSE"],
    ):
        sub = dash_df[dash_df["task"] == task_name].copy()
        if sub.empty:
            ax.set_visible(False)
            continue

        score_col = f"mean_{sub['ranking_metric'].iloc[0]}"
        targets = sorted(sub["target_label"].unique())
        horizons_here = sorted(sub["horizon"].dropna().unique())
        n_h = len(horizons_here)
        width = 0.8 / max(n_h, 1)
        x = np.arange(len(targets))

        for i, h in enumerate(horizons_here):
            vals, colors, feats = [], [], []
            for t in targets:
                row = sub[(sub["target_label"] == t) & (sub["horizon"] == h)]
                if row.empty:
                    vals.append(np.nan); colors.append("lightgray"); feats.append("")
                else:
                    vals.append(row[score_col].iloc[0])
                    colors.append(color_for_model(row["model"].iloc[0]))
                    feats.append(row["top_feature"].iloc[0] if pd.notna(row["top_feature"].iloc[0]) else "")

            bars = ax.bar(x + i * width, vals, width, color=colors, edgecolor="black", linewidth=0.4)
            valid_vals = [v for v in vals if pd.notna(v)]
            ymax_local = max(valid_vals) if valid_vals else 1
            for xb, v, f in zip(x + i * width, vals, feats):
                if pd.notna(v) and f:
                    ax.text(xb, v + 0.01 * ymax_local, f, rotation=90, ha="center", va="bottom", fontsize=6)

        ax.set_xticks(x + width * (n_h - 1) / 2)
        ax.set_xticklabels(targets, rotation=30, ha="right", fontsize=8)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(task_name.capitalize(), fontsize=12)
        ax.set_ylim(top=ax.get_ylim()[1] * 1.25)

    fig.legend(
        handles=category_legend_handles(), loc="lower center", ncol=3,
        bbox_to_anchor=(0.5, -0.06),
    )
    fig.suptitle(
        "Best Model (bar color) and Its Top Driving Feature (label) by Target & Horizon",
        fontsize=14, y=1.03,
    )

    fig.tight_layout(rect=[0, 0.04, 1, 1])
    save_fig(fig, AGG_ROOT, "best_model_and_top_feature_dashboard")
    plt.show()

    print(
        "Suggested caption: \"Winning model's mean TEST performance (bar height/color) and "
        "its single most important feature by SHAP (label) for every forecasting target and "
        "horizon.\"\n"
    )


### 13.6 Publication-ready summary table (new)

A single, presentation-ready version of `best_model_feature_summary`, with the numeric
column rounded and background shading on the "Top Domain Share" column so higher-coverage
domains stand out — meant to be screenshotted straight into a paper/slide, or exported via
`.to_latex()` the same way Section 11 (LaTeX export cell) already does for the leaderboard
tables.

In [ ]:
# ------------------------------------------------------------------
# 13.6 Styled version of best_model_feature_summary for direct use in
# the paper / a slide (no new data -- purely a presentation layer).
# ------------------------------------------------------------------

if best_model_feature_summary.empty:
    print("best_model_feature_summary is empty.")
else:
    styled_summary = (
        best_model_feature_summary
        .style
        .background_gradient(subset=["Top Domain Share"], cmap="Blues")
        .format({"Top Domain Share": "{:.3f}"})
        .set_caption("Best model + top domain + top-3 features, by target and horizon")
    )
    display(styled_summary)


### Notes (Section 13)

- `top_features_for_layer_horizon` and `top_domain_for_layer_horizon` reuse
  `feature_importance_avg` / `domain_importance_tables` from Section 12 --
  no feature-importance data is recomputed here.
- As in Section 9b, a feature/domain is only reported if it has data for
  **every** symbol at that (layer, horizon); partial coverage is dropped
  rather than silently averaged over fewer symbols.
- `detail_layer` / `detail_horizon` (13.2) and `grid_layer` (13.3) are the
  only variables you should need to change to explore a different
  target/horizon.
- Because `shap_importance_test` is target/feature-level (not per-model,
  see Section 12.3), the "top features" shown here explain the *target*,
  not the specific winning model's decision process.


## 14. Notebook-wide sanity checks (new)

A short, explicit audit of things that could silently make the plots above misleading,
run last so it has access to every DataFrame built earlier in the notebook:

1. **Coverage** — does every (target, horizon) cell in the best-model summary really rest
   on all three indices (already checked once for Section 11 in 11.9; repeated here for
   the feature-importance side too)?
2. **High-uncertainty wins** — any (target, horizon) where the winning model's cross-index
   standard deviation is larger than a reasonable fraction of the score itself, meaning
   the "win" is not very robust across NIFTY50 / BANKNIFTY / NIFTYNEXT50.
3. **Excluded models** — a reminder of which baselines were dropped everywhere (`Naive`,
   `ARIMA`) and how many rows that removed, so it's on record for the methodology section.
4. **Output inventory** — a listing of every file this notebook wrote to
   `AGG_ROOT` / `FEATURE_AGG_ROOT`, so it's easy to check nothing is missing before
   assembling the paper's figures.

In [ ]:
# ------------------------------------------------------------------
# 14.1 Coverage check -- feature-importance side
# ------------------------------------------------------------------
expected_cells = len(LAYER_CONFIG) * len(HORIZONS)
fi_have = 0
fi_missing = []
for layer in LAYER_CONFIG:
    for h in HORIZONS:
        x = feature_importance_avg[(feature_importance_avg['layer'] == layer) & (feature_importance_avg['horizon'] == h)]
        n_syms = x['symbol'].nunique() if not x.empty else 0
        if n_syms >= len(SYMBOLS):
            fi_have += 1
        else:
            fi_missing.append((layer, h, n_syms))

print(f"Feature-importance coverage: {fi_have} / {expected_cells} target x horizon cells have all {len(SYMBOLS)} symbols.")
if fi_missing:
    print("Incomplete cells:")
    for layer, h, n in fi_missing:
        print(f"  {layer}  h={h}  ({n}/{len(SYMBOLS)} symbols)")

# ------------------------------------------------------------------
# 14.2 High-uncertainty wins -- SD is a large fraction of the score
# ------------------------------------------------------------------
print("\nWins where the cross-index standard deviation exceeds 15% of the mean score:")
if not best_model_summary.empty:
    flagged_rows = []
    for _, row in best_model_summary.iterrows():
        rank_metric = row['ranking_metric']
        mean_val = row.get(f'mean_{rank_metric}')
        std_val = row.get(f'std_{rank_metric}')
        if pd.notna(mean_val) and pd.notna(std_val) and mean_val != 0 and (std_val / abs(mean_val)) > 0.15:
            flagged_rows.append({
                'layer': row['layer'], 'horizon': row['horizon'], 'model': row['model'],
                'mean': round(mean_val, 4), 'std': round(std_val, 4),
                'cv_pct': round(100 * std_val / abs(mean_val), 1),
            })
    if flagged_rows:
        display(pd.DataFrame(flagged_rows).sort_values('cv_pct', ascending=False))
    else:
        print("  None -- every generalized winner is reasonably stable across indices.")
else:
    print("  best_model_summary is empty.")

# ------------------------------------------------------------------
# 14.3 Excluded-model reminder
# ------------------------------------------------------------------
print(f"\nModels excluded from every table/plot in this notebook: {sorted(EXCLUDED_MODELS)}")

# ------------------------------------------------------------------
# 14.4 Output inventory
# ------------------------------------------------------------------
print("\nFiles written to AGG_ROOT (leaderboard outputs):")
for p in sorted(Path(AGG_ROOT).glob('*')):
    print(' ', p.name)

print("\nFiles written to FEATURE_AGG_ROOT (feature-importance outputs):")
for p in sorted(Path(FEATURE_AGG_ROOT).glob('*')):
    print(' ', p.name)
